## Match Top Monthly Physical Books by Checkout Volume to Ebook Checkouts to see if popularity persists cross-channel

### Setup data

In [1]:
## Import packages and datasets
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import numpy as np
from rapidfuzz import process, fuzz
import re

processed_dir = '/Users/audriswong/data-portfolio/projects/seattle-checkouts/data/processed'

book_checkouts = pd.read_parquet(f'{processed_dir}/book_checkouts.parquet').rename(columns={'title_clean': 'title'}) 
ebook_checkouts = pd.read_parquet(f'{processed_dir}/ebook_checkouts.parquet').rename(columns={'title_clean': 'title'})

In [2]:
#rank add monthly checkout ranking per work to all titles

def works_with_monthly_rank(df, year_col='checkoutyear', month_col='checkoutmonth'):
    results = []
    for (year, month), group in df.groupby([year_col, month_col]):
        ranked = (group.groupby('work_key')
                .agg(total_checkouts=('checkouts', 'sum'),
                     genre=('genre', 'first'),
                     title=('title', 'first'),
                     creator=('creator_clean', 'first'),
                     month_date = ('month_date', 'first'))
                .reset_index()
                .sort_values('total_checkouts', ascending=False))
        ranked['month_checkout_rank'] = range(1, len(ranked) + 1)
        ranked['checkoutyear']  = year
        ranked['checkoutmonth'] = month
        results.append(ranked)
    return pd.concat(results, ignore_index=True)

book_ranked  = works_with_monthly_rank(book_checkouts)
ebook_ranked = works_with_monthly_rank(ebook_checkouts)

In [3]:
print("=== Book_Ranked sample ===")
print(book_ranked.head())

print("\n=== EBook_ranked sample ===")
print(ebook_ranked.head())

=== Book_Ranked sample ===
                                            work_key  total_checkouts  \
0  TODAY WILL BE DIFFERENT : A NOVEL || SEMPLE, M...              898   
1  HILLBILLY ELEGY : A MEMOIR OF A FAMILY AND CUL...              596   
2  THE UNDERGROUND RAILROAD : A NOVEL || WHITEHEA...              542   
3              MOONGLOW : A NOVEL || CHABON, MICHAEL              442   
4  THE WRONG SIDE OF GOODBYE : A NOVEL || CONNELL...              400   

              genre                                              title  \
0             Humor                  TODAY WILL BE DIFFERENT : A NOVEL   
1  Biography/Memoir  HILLBILLY ELEGY : A MEMOIR OF A FAMILY AND CUL...   
2        Historical                 THE UNDERGROUND RAILROAD : A NOVEL   
3           Fiction                                 MOONGLOW : A NOVEL   
4  Mystery/Thriller                THE WRONG SIDE OF GOODBYE : A NOVEL   

             creator month_date  month_checkout_rank  checkoutyear  \
0      SEMPLE, MARI

### Develop Work Matching Logic

In [5]:
#method 1: match using exact work_key match - ebook and book work_key have to be identical
 
# Filter book_ranked to top 5 per month
book_top5_ranked = book_ranked[book_ranked['month_checkout_rank'] <= 5][
    ['work_key', 'title', 'creator', 'checkoutyear', 'checkoutmonth', 'genre', 'total_checkouts', 'month_checkout_rank']
].rename(columns={
    'total_checkouts':    'book_total_checkouts',
    'month_checkout_rank': 'book_rank'
})

# Join to ebook_ranked on work_key + month
crossed = book_top5_ranked.merge(
    ebook_ranked[['work_key', 'title', 'creator', 'checkoutyear', 'checkoutmonth', 'total_checkouts', 'month_checkout_rank']].rename(columns={
        'total_checkouts':    'ebook_total_checkouts',
        'month_checkout_rank': 'ebook_rank'
    }),
    on=['work_key', 'checkoutyear', 'checkoutmonth'],
    how='left'
)

# Flag ebook rank tiers
crossed['in_ebook_top5']  = crossed['ebook_rank'] <= 5
crossed['in_ebook_top10'] = crossed['ebook_rank'] <= 10
crossed['in_ebook_top20'] = crossed['ebook_rank'] <= 20
crossed['in_ebook_top50'] = crossed['ebook_rank'] <= 50

# Summary — how often do top 5 books appear in each ebook tier?
print("=== How often do top 5 physical books appear in ebook rankings? ===")
print(f"In ebook top 5:  {crossed['in_ebook_top5'].sum()} months")
print(f"In ebook top 10: {crossed['in_ebook_top10'].sum()} months")
print(f"In ebook top 20: {crossed['in_ebook_top20'].sum()} months")
print(f"In ebook top 50: {crossed['in_ebook_top50'].sum()} months")
print(f"Not in ebook rankings at all: {crossed['ebook_rank'].isna().sum()} months")

# View the actual matches
print("\n=== Works that appear in both top 5 books AND top 50 ebooks ===")
print(crossed[crossed['in_ebook_top50']].sort_values(['checkoutyear', 'checkoutmonth', 'book_rank'])
      .to_string(index=False))


=== How often do top 5 physical books appear in ebook rankings? ===
In ebook top 5:  0 months
In ebook top 10: 0 months
In ebook top 20: 0 months
In ebook top 50: 0 months
Not in ebook rankings at all: 540 months

=== Works that appear in both top 5 books AND top 50 ebooks ===
Empty DataFrame
Columns: [work_key, title_x, creator_x, checkoutyear, checkoutmonth, genre, book_total_checkouts, book_rank, title_y, creator_y, ebook_total_checkouts, ebook_rank, in_ebook_top5, in_ebook_top10, in_ebook_top20, in_ebook_top50]
Index: []


In [6]:
## method 2: fuzzy match on work_key with .8+ score
from rapidfuzz import process, fuzz

def fuzzy_match_work_key(book_key, ebook_keys, threshold=80):
    """Find best matching ebook work_key for a given book work_key"""
    match = process.extractOne(book_key, ebook_keys, 
                                scorer=fuzz.token_sort_ratio,
                                score_cutoff=threshold)
    if match:
        return match[0], match[1]  # matched key, score
    return None, None

# Get unique ebook work_keys per month for faster lookup
def fuzzy_cross_rank(book_df, ebook_df, book_rank_cutoff=5, threshold=80):
    results = []
    
    for (year, month), book_group in book_df[book_df['month_checkout_rank'] <= book_rank_cutoff].groupby(['checkoutyear', 'checkoutmonth']):
        ebook_month = ebook_df[(ebook_df['checkoutyear'] == year) & 
                                (ebook_df['checkoutmonth'] == month)]
        ebook_keys = ebook_month['work_key'].tolist()
        
        for _, book_row in book_group.iterrows():
            matched_key, score = fuzzy_match_work_key(book_row['work_key'], ebook_keys, threshold)
            
            if matched_key:
                ebook_row = ebook_month[ebook_month['work_key'] == matched_key].iloc[0]
                results.append({
                    'checkoutyear':        year,
                    'checkoutmonth':       month,
                    'book_work_key':       book_row['work_key'],
                    'ebook_work_key':      matched_key,
                    'match_score':         score,
                    'genre':               book_row['genre'],
                    'book_rank':           book_row['month_checkout_rank'],
                    'book_total_checkouts': book_row['total_checkouts'],
                    'ebook_rank':          ebook_row['month_checkout_rank'],
                    'ebook_total_checkouts': ebook_row['total_checkouts'],
                })
            else:
                results.append({
                    'checkoutyear':        year,
                    'checkoutmonth':       month,
                    'book_work_key':       book_row['work_key'],
                    'ebook_work_key':      None,
                    'match_score':         None,
                    'genre':               book_row['genre'],
                    'book_rank':           book_row['month_checkout_rank'],
                    'book_total_checkouts': book_row['total_checkouts'],
                    'ebook_rank':          None,
                    'ebook_total_checkouts': None,
                })
    
    return pd.DataFrame(results)

fuzzy_crossed = fuzzy_cross_rank(book_ranked, ebook_ranked, book_rank_cutoff=5, threshold=80)

print("=== Sample Matched Results ===")
print(fuzzy_crossed.sort_values(['checkoutyear', 'checkoutmonth', 'book_rank']).head().to_string(index=False))

total = len(fuzzy_crossed)
matched = fuzzy_crossed['ebook_rank'].notna().sum()

print("=== Method 2: How often do top 5 physical books appear in ebook rankings? ===")
print(f"Total book entries:    {total:,}")
print(f"Matched in ebooks:     {matched:,} ({matched/total*100:.1f}%)")
print(f"Not matched in ebooks: {total-matched:,} ({(total-matched)/total*100:.1f}%)")
print()

for tier in [5, 10, 20, 50]:
    count = (fuzzy_crossed['ebook_rank'] <= tier).sum()
    print(f"In ebook top {tier:>2}: {count:>5,} ({count/total*100:.1f}%)")



=== Sample Matched Results ===
 checkoutyear  checkoutmonth                                                                book_work_key                                                             ebook_work_key  match_score            genre  book_rank  book_total_checkouts  ebook_rank  ebook_total_checkouts
         2017              1                           TODAY WILL BE DIFFERENT : A NOVEL || SEMPLE, MARIA                                    TODAY WILL BE DIFFERENT || MARIA SEMPLE    87.640449            Humor          1                   898         3.0                  414.0
         2017              1 HILLBILLY ELEGY : A MEMOIR OF A FAMILY AND CULTURE IN CRISIS || VANCE, J. D. HILLBILLY ELEGY: A MEMOIR OF A FAMILY AND CULTURE IN CRISIS || J. D. VANCE    97.333333 Biography/Memoir          2                   596        24.0                  229.0
         2017              1                      THE UNDERGROUND RAILROAD : A NOVEL || WHITEHEAD, COLSON  THE UNDERGROUND RAILROAD 

In [7]:
#are there any noise words commonly found within the unmatched works?

from collections import Counter
import re

# Get unmatched book work keys
unmatched_keys = fuzzy_crossed[fuzzy_crossed['ebook_work_key'].isna()]['book_work_key'].tolist()

# Extract individual words and short phrases
word_counts = Counter()
bigram_counts = Counter()

for key in unmatched_keys:
    words = re.sub(r'[^a-z0-9\s]', ' ', key.lower()).split()
    # Single words
    word_counts.update(words)
    # Bigrams (2-word phrases)
    bigrams = [f"{words[i]} {words[i+1]}" for i in range(len(words)-1)]
    bigram_counts.update(bigrams)

print("=== Most common single words in unmatched titles ===")
for word, count in word_counts.most_common(20):
    print(f"{count:>5} | {word}")

print("\n=== Most common 2-word phrases in unmatched titles ===")
for bigram, count in bigram_counts.most_common(20):
    print(f"{count:>5} | {bigram}")

=== Most common single words in unmatched titles ===
   35 | the
   27 | a
   21 | novel
   12 | of
    8 | lee
    8 | jane
    7 | child
    7 | elizabeth
    6 | louise
    5 | connelly
    5 | michael
    5 | penny
    5 | and
    5 | ann
    4 | harper
    4 | big
    4 | tom
    4 | lake
    4 | patchett
    3 | wrong

=== Most common 2-word phrases in unmatched titles ===
   19 | a novel
    7 | child lee
    5 | connelly michael
    5 | penny louise
    5 | of the
    4 | harper jane
    4 | jane jane
    4 | jane elizabeth
    4 | tom lake
    4 | lake a
    4 | novel patchett
    4 | patchett ann
    3 | the wrong
    3 | wrong side
    3 | side of
    3 | of goodbye
    3 | goodbye a
    3 | novel connelly
    3 | less a
    3 | novel greer


In [8]:
sample_work = book_ranked['work_key'].iloc[0]
sample_work_cleaned = sample_work.lower().replace('novel', '').upper()

print(sample_work_cleaned)

print('Uncleaned Matches:')
print(process.extract(sample_work, ebook_ranked['work_key'].unique().tolist()) )

print('Cleaned Matches:')
print(process.extract(sample_work_cleaned, ebook_ranked['work_key'].unique().tolist()) )


TODAY WILL BE DIFFERENT : A  || SEMPLE, MARIA
Uncleaned Matches:
[('TODAY WILL BE DIFFERENT || MARIA SEMPLE', 85.6338028169014, 2), ('COMMONWEALTH || ANN PATCHETT', 85.5, 3), ('THE MARTIAN: A NOVEL || ANDY WEIR', 85.5, 5), ('THE LIFE-CHANGING MAGIC OF TIDYING UP: THE JAPANESE ART OF DECLUTTERING AND ORGANIZING || MARIE KONDO', 85.5, 6), ('ME BEFORE YOU || JOJO MOYES', 85.5, 7)]
Cleaned Matches:
[('TODAY WILL BE DIFFERENT || MARIA SEMPLE', 89.27710843373494, 2), ("THE UNDERGROUND RAILROAD (OPRAH'S BOOK CLUB): A NOVEL || COLSON WHITEHEAD", 85.5, 1), ('COMMONWEALTH || ANN PATCHETT', 85.5, 3), ('THE LIFE-CHANGING MAGIC OF TIDYING UP: THE JAPANESE ART OF DECLUTTERING AND ORGANIZING || MARIE KONDO', 85.5, 6), ('ME BEFORE YOU || JOJO MOYES', 85.5, 7)]


In [9]:
#method 3: fuzzy match on title score .8+, use author fuzzy match as tiebreaker if multiple matches

def fuzzy_cross_rank_title_author(book_df, ebook_df, book_rank_cutoff=5, title_threshold=80, author_threshold=80):
    results = []
    
    for (year, month), book_group in book_df[book_df['month_checkout_rank'] <= book_rank_cutoff].groupby(['checkoutyear', 'checkoutmonth']):
        ebook_month = ebook_df[(ebook_df['checkoutyear'] == year) & 
                                (ebook_df['checkoutmonth'] == month)].drop_duplicates('title')
        
        ebook_titles  = ebook_month['title'].tolist()
        ebook_authors = ebook_month['creator'].tolist()

        for _, book_row in book_group.iterrows():
            # Step 1 — find top title candidates above threshold
            title_candidates = process.extract(book_row['title'], ebook_titles,
                                               scorer=fuzz.token_sort_ratio, limit=5)
            
            # Filter to candidates that pass title threshold
            passing = [(title, score, idx) for title, score, idx in title_candidates
                       if score >= title_threshold]
            
            best_match      = None
            best_title_score  = None
            best_author_score = None

            if len(passing) == 0:
                # No title matches
                best_match = None

            elif len(passing) == 1:
                # Only one match — accept it without checking author
                _, best_title_score, idx = passing[0]
                best_match        = ebook_month.iloc[idx]
                best_author_score = None

            else:
                # Multiple title matches — use author as tiebreaker
                best_combined_score = 0
                for _, title_score, idx in passing:
                    ebook_author = ebook_authors[idx]
                    author_score = fuzz.token_sort_ratio(book_row['creator'], ebook_author)
                    combined_score = (title_score * 0.7) + (author_score * 0.3)
                    
                    if combined_score > best_combined_score:
                        best_combined_score = combined_score
                        best_match        = ebook_month.iloc[idx]
                        best_title_score  = title_score
                        best_author_score = author_score

            if best_match is not None:
                results.append({
                    'checkoutyear':          year,
                    'checkoutmonth':         month,
                    'book_title':            book_row['title'],
                    'ebook_title':           best_match['title'],
                    'book_author':           book_row['creator'],
                    'ebook_author':          best_match['creator'],
                    'title_score':           best_title_score,
                    'author_score':          best_author_score,
                    'genre':                 book_row['genre'],
                    'book_rank':             book_row['month_checkout_rank'],
                    'book_total_checkouts':  book_row['total_checkouts'],
                    'ebook_rank':            best_match['month_checkout_rank'],
                    'ebook_total_checkouts': best_match['total_checkouts'],
                })
            else:
                results.append({
                    'checkoutyear':          year,
                    'checkoutmonth':         month,
                    'book_title':            book_row['title'],
                    'ebook_title':           None,
                    'book_author':           book_row['creator'],
                    'ebook_author':          None,
                    'title_score':           None,
                    'author_score':          None,
                    'genre':                 book_row['genre'],
                    'book_rank':             book_row['month_checkout_rank'],
                    'book_total_checkouts':  book_row['total_checkouts'],
                    'ebook_rank':            None,
                    'ebook_total_checkouts': None,
                })
    
    return pd.DataFrame(results)

fuzzy_crossed_v2 = fuzzy_cross_rank_title_author(book_ranked, ebook_ranked, book_rank_cutoff=5)
print(fuzzy_crossed_v2.sort_values(['checkoutyear', 'checkoutmonth', 'book_rank']).head().to_string(index=False))

total = len(fuzzy_crossed_v2)
matched = fuzzy_crossed_v2['ebook_rank'].notna().sum()

print("\n=== Method 3: How often do top 5 physical books appear in ebook rankings with title + author tiebreak matching? ===")
print(f"Total book entries:    {total:,}")
print(f"Matched in ebooks:     {matched:,} ({matched/total*100:.1f}%)")
print(f"Not matched in ebooks: {total-matched:,} ({(total-matched)/total*100:.1f}%)")
print()

for tier in [5, 10, 20, 50]:
    count = (fuzzy_crossed_v2['ebook_rank'] <= tier).sum()
    print(f"In ebook top {tier:>2}: {count:>5,} ({count/total*100:.1f}%)")

 checkoutyear  checkoutmonth                                                   book_title                                                 ebook_title       book_author   ebook_author  title_score  author_score            genre  book_rank  book_total_checkouts  ebook_rank  ebook_total_checkouts
         2017              1                            TODAY WILL BE DIFFERENT : A NOVEL                                     TODAY WILL BE DIFFERENT     SEMPLE, MARIA   MARIA SEMPLE    82.142857           NaN            Humor          1                   898         3.0                  414.0
         2017              1 HILLBILLY ELEGY : A MEMOIR OF A FAMILY AND CULTURE IN CRISIS HILLBILLY ELEGY: A MEMOIR OF A FAMILY AND CULTURE IN CRISIS      VANCE, J. D.    J. D. VANCE    97.478992           NaN Biography/Memoir          2                   596        24.0                  229.0
         2017              1                           THE UNDERGROUND RAILROAD : A NOVEL                          

In [10]:
##step by step debug of unmatched titles

from rapidfuzz import process, fuzz

# Pick a specific month to debug
year, month = 2017, 1

# Get unmatched book titles for that month from fuzzy_crossed_v2
unmatched_titles = fuzzy_crossed_v2[
    (fuzzy_crossed_v2['checkoutyear'] == year) &
    (fuzzy_crossed_v2['checkoutmonth'] == month) &
    (fuzzy_crossed_v2['ebook_rank'].isna())
]['book_title'].tolist()

# Look up those titles in book_ranked
book_sample = book_ranked[
    (book_ranked['checkoutyear'] == year) &
    (book_ranked['checkoutmonth'] == month) &
    (book_ranked['title'].isin(unmatched_titles))
]

# All ebooks for that month — no filtering needed
ebook_sample = ebook_ranked[
    (ebook_ranked['checkoutyear'] == year) &
    (ebook_ranked['checkoutmonth'] == month)
].drop_duplicates('title')

print("=== Top 5 book titles ===")
print(book_sample[['title', 'creator']].to_string(index=False))

print("\n=== Top 10 ebook titles ===")
print(ebook_sample[['title', 'creator']].head(10).to_string(index=False))

print("\n=== Fuzzy match results on work key for each book title ===")
for _, row in book_sample.iterrows():
    print(f"\nBook: {row['work_key']}")
    matches = process.extract(row['work_key'], ebook_sample['work_key'].tolist(),
                               scorer=fuzz.token_sort_ratio, limit=5)
    for matched_work_key, score, idx in matches:
        ebook_work_key = ebook_sample['work_key'].tolist()[idx]
        print(f"  Score: {score:>5.1f} | {matched_work_key} ")

print("\n=== Fuzzy match results on title + author (tiebreak) for each book title ===")
for _, row in book_sample.iterrows():
    print(f"\nBook: {row['title']} | {row['creator']}")
    matches = process.extract(row['title'], ebook_sample['title'].tolist(),
                               scorer=fuzz.token_sort_ratio, limit=5)
    for matched_title, score, idx in matches:
        ebook_author = ebook_sample['creator'].tolist()[idx]
        author_score = fuzz.token_sort_ratio(row['creator'], ebook_author)
        print(f"  Score: {score:>5.1f} | Author: {author_score:>5.1f} | {matched_title} | {ebook_author}")

=== Top 5 book titles ===
                              title           creator
 THE UNDERGROUND RAILROAD : A NOVEL WHITEHEAD, COLSON
THE WRONG SIDE OF GOODBYE : A NOVEL CONNELLY, MICHAEL

=== Top 10 ebook titles ===
                                                                                 title          creator
                                   THE GOLDFINCH: A NOVEL (PULITZER PRIZE FOR FICTION)      DONNA TARTT
                                 THE UNDERGROUND RAILROAD (OPRAH'S BOOK CLUB): A NOVEL COLSON WHITEHEAD
                                                               TODAY WILL BE DIFFERENT     MARIA SEMPLE
                                                                          COMMONWEALTH     ANN PATCHETT
                                                              BETWEEN THE WORLD AND ME TA-NEHISI COATES
                                                                  THE MARTIAN: A NOVEL        ANDY WEIR
THE LIFE-CHANGING MAGIC OF TIDYING UP: THE JAPANESE ART

In [11]:
#method 4 - Fuzzy Match on title, break match ties on author
from rapidfuzz import process, fuzz

def fuzzy_cross_rank_title_author(book_df, ebook_df, book_rank_cutoff=5, tie_threshold=2):
    """
    Match on title score — take the best match.
    Use author as tiebreaker if top scores are within tie_threshold points of each other.
    """
    results = []
    
    for (year, month), book_group in book_df[book_df['month_checkout_rank'] <= book_rank_cutoff].groupby(['checkoutyear', 'checkoutmonth']):
        ebook_month = ebook_df[(ebook_df['checkoutyear'] == year) & 
                                (ebook_df['checkoutmonth'] == month)].drop_duplicates('title')
        
        ebook_titles  = ebook_month['title'].tolist()
        ebook_authors = ebook_month['creator'].tolist()

        for _, book_row in book_group.iterrows():
            title_candidates = process.extract(book_row['title'], ebook_titles,
                                               scorer=fuzz.token_sort_ratio, limit=5)
            
            if not title_candidates:
                best_match        = None
                best_title_score  = None
                best_author_score = None
            else:
                top_score = title_candidates[0][1]
                
                # Find all candidates within tie_threshold of the top score
                tied = [(title, score, idx) for title, score, idx in title_candidates
                        if top_score - score <= tie_threshold]
                
                if len(tied) == 1:
                    # Clear winner — take it without checking author
                    _, best_title_score, idx = tied[0]
                    best_match        = ebook_month.iloc[idx]
                    best_author_score = None
                else:
                    # Tied — use author as tiebreaker
                    best_combined_score = 0
                    best_match          = None
                    best_title_score    = None
                    best_author_score   = None
                    for _, title_score, idx in tied:
                        author_score   = fuzz.token_sort_ratio(book_row['creator'], ebook_authors[idx])
                        combined_score = (title_score * 0.7) + (author_score * 0.3)
                        if combined_score > best_combined_score:
                            best_combined_score = combined_score
                            best_match          = ebook_month.iloc[idx]
                            best_title_score    = title_score
                            best_author_score   = author_score

            if best_match is not None:
                results.append({
                    'checkoutyear':          year,
                    'checkoutmonth':         month,
                    'book_title':            book_row['title'],
                    'ebook_title':           best_match['title'],
                    'book_author':           book_row['creator'],
                    'ebook_author':          best_match['creator'],
                    'title_score':           best_title_score,
                    'author_score':          best_author_score,
                    'genre':                 book_row['genre'],
                    'book_rank':             book_row['month_checkout_rank'],
                    'book_total_checkouts':  book_row['total_checkouts'],
                    'ebook_rank':            best_match['month_checkout_rank'],
                    'ebook_total_checkouts': best_match['total_checkouts'],
                })
            else:
                results.append({
                    'checkoutyear':          year,
                    'checkoutmonth':         month,
                    'book_title':            book_row['title'],
                    'ebook_title':           None,
                    'book_author':           book_row['creator'],
                    'ebook_author':          None,
                    'title_score':           None,
                    'author_score':          None,
                    'genre':                 book_row['genre'],
                    'book_rank':             book_row['month_checkout_rank'],
                    'book_total_checkouts':  book_row['total_checkouts'],
                    'ebook_rank':            None,
                    'ebook_total_checkouts': None,
                })
    
    return pd.DataFrame(results)

fuzzy_crossed_v3 = fuzzy_cross_rank_title_author(book_ranked, ebook_ranked, book_rank_cutoff=5)

#this should by default match every book - confirm
total = len(fuzzy_crossed_v3)
matched = fuzzy_crossed_v3['ebook_rank'].notna().sum()

print("\n=== Method 4: How often do top 5 physical books appear in ebook rankings with closest title + author tiebreak matching? ===")
print(f"Total book entries:    {total:,}")
print(f"Matched in ebooks:     {matched:,} ({matched/total*100:.1f}%)")
print(f"Not matched in ebooks: {total-matched:,} ({(total-matched)/total*100:.1f}%)")
print()

for tier in [5, 10, 20, 50]:
    count = (fuzzy_crossed_v3['ebook_rank'] <= tier).sum()
    print(f"In ebook top {tier:>2}: {count:>5,} ({count/total*100:.1f}%)")


=== Method 4: How often do top 5 physical books appear in ebook rankings with closest title + author tiebreak matching? ===
Total book entries:    540
Matched in ebooks:     540 (100.0%)
Not matched in ebooks: 0 (0.0%)

In ebook top  5:    57 (10.6%)
In ebook top 10:    81 (15.0%)
In ebook top 20:   135 (25.0%)
In ebook top 50:   204 (37.8%)


In [12]:
# validate
# spot check the 10 lowest scoring matches
worst_matches = fuzzy_crossed_v3.nsmallest(10, 'title_score')[['book_title', 'ebook_title', 'book_author', 'ebook_author', 'title_score', 'author_score']]

print("=== 10 Lowest Title Scoring Matches ===")
print(worst_matches.to_string(index=False))

# For each worst match, show all 5 candidates that were considered
print("\n=== All candidates considered for each match ===")
for _, row in worst_matches.iterrows():
    print(f"\nBook:  {row['book_title']} | {row['book_author']}")
    print(f"Chose: {row['ebook_title']} | {row['ebook_author']} (score: {row['title_score']})")
    
    # Get the ebook titles for that month
    year  = fuzzy_crossed_v3[fuzzy_crossed_v3['book_title'] == row['book_title']]['checkoutyear'].iloc[0]
    month = fuzzy_crossed_v3[fuzzy_crossed_v3['book_title'] == row['book_title']]['checkoutmonth'].iloc[0]
    
    ebook_month  = ebook_ranked[(ebook_ranked['checkoutyear'] == year) &
                                 (ebook_ranked['checkoutmonth'] == month)].drop_duplicates('title')
    ebook_titles = ebook_month['title'].tolist()
    
    candidates = process.extract(row['book_title'], ebook_titles, scorer=fuzz.token_sort_ratio, limit=5)
    print("  All candidates:")
    for matched_title, score, idx in candidates:
        author_score = fuzz.token_sort_ratio(row['book_author'], ebook_month['creator'].tolist()[idx])
        print(f"  {score:>5.1f} title | {author_score:>5.1f} author | {matched_title}")

=== 10 Lowest Title Scoring Matches ===
                                                                                                                                                  book_title                                                                                                                       ebook_title            book_author      ebook_author  title_score  author_score
                                                                              SELOC HONDA OUTBOARDS : 1978-99 REPAIR MANUAL, 2-130 HORSEPOWER, 1-4 CYCLINDER                                                                                  STITCHES: A HANDBOOK ON MEANING, HOPE AND REPAIR                   None       ANNE LAMOTT    46.031746     13.333333
                                                                                    SEMBRANDO HISTORIAS : PURA BELPRÉ : BIBLIOTECARIA Y NARRADORA DE CUENTOS                                                                                              

In [13]:
#the lowest 10 match scores are wrong.  Validate this across the entire dataset

def validate_match(book_title, ebook_title, book_author, ebook_author, 
                   author_threshold=70, min_substring_len=3):
    """
    Validate a fuzzy match by:
    1. Check if authors match above threshold
    2. If yes, check if any meaningful word from book title appears in ebook title
    """
    # Step 1 — author match
    author_score = fuzz.token_sort_ratio(str(book_author), str(ebook_author))
    if author_score < author_threshold:
        return False, author_score, None
    
    # Step 2 — substring check: any word from book title in ebook title?
    book_words = set(re.sub(r'[^a-z0-9\s]', ' ', str(book_title).lower()).split())
    ebook_title_lower = str(ebook_title).lower()
    
    # Filter out noise words
    noise = {'a', 'an', 'the', 'of', 'in', 'and', 'or', 'to', 'by', 'for', 
             'novel', 'memoir', 'story', 'book', 'vol', 'volume'}
    meaningful_words = [w for w in book_words if w not in noise and len(w) >= min_substring_len]
    
    matched_words = [w for w in meaningful_words if w in ebook_title_lower]
    
    if matched_words:
        return True, author_score, matched_words
    return False, author_score, None

# Apply validation to fuzzy_crossed_v3
fuzzy_crossed_v3['valid_match'] = False
fuzzy_crossed_v3['validated_author_score'] = None
fuzzy_crossed_v3['matched_words'] = None

for idx, row in fuzzy_crossed_v3[fuzzy_crossed_v3['ebook_title'].notna()].iterrows():
    is_valid, author_score, matched_words = validate_match(
        row['book_title'], row['ebook_title'],
        row['book_author'], row['ebook_author']
    )
    fuzzy_crossed_v3.at[idx, 'valid_match']           = is_valid
    fuzzy_crossed_v3.at[idx, 'validated_author_score'] = author_score
    fuzzy_crossed_v3.at[idx, 'matched_words']          = str(matched_words) if matched_words else None

# Summary
total    = len(fuzzy_crossed_v3)
matched  = fuzzy_crossed_v3['ebook_title'].notna().sum()
valid    = fuzzy_crossed_v3['valid_match'].sum()
invalid  = matched - valid

print("=== Match Validation Summary ===")
print(f"Total book entries:       {total:,}")
print(f"Fuzzy matched:            {matched:,} ({matched/total*100:.1f}%)")
print(f"Validated (author+word):  {valid:,} ({valid/total*100:.1f}%)")
print(f"Invalid matches:          {invalid:,} ({invalid/total*100:.1f}%)")
print(f"No match found:           {total-matched:,} ({(total-matched)/total*100:.1f}%)")

print("\n=== Sample valid matches ===")
print(fuzzy_crossed_v3[fuzzy_crossed_v3['valid_match']][
    ['book_title', 'ebook_title', 'book_author', 'ebook_author', 'validated_author_score', 'matched_words']
].head(10).to_string(index=False))

print("\n=== Sample invalid matches ===")
print(fuzzy_crossed_v3[(fuzzy_crossed_v3['ebook_title'].notna()) & 
                        (~fuzzy_crossed_v3['valid_match'])][
    ['book_title', 'ebook_title', 'book_author', 'ebook_author', 'validated_author_score', 'matched_words']
].head(10).to_string(index=False))

=== Match Validation Summary ===
Total book entries:       540
Fuzzy matched:            540 (100.0%)
Validated (author+word):  369 (68.3%)
Invalid matches:          171 (31.7%)
No match found:           0 (0.0%)

=== Sample valid matches ===
                                                  book_title                                                 ebook_title       book_author     ebook_author validated_author_score                                         matched_words
                           TODAY WILL BE DIFFERENT : A NOVEL                                     TODAY WILL BE DIFFERENT     SEMPLE, MARIA     MARIA SEMPLE                   96.0                        ['today', 'different', 'will']
HILLBILLY ELEGY : A MEMOIR OF A FAMILY AND CULTURE IN CRISIS HILLBILLY ELEGY: A MEMOIR OF A FAMILY AND CULTURE IN CRISIS      VANCE, J. D.      J. D. VANCE              95.652174 ['crisis', 'elegy', 'family', 'hillbilly', 'culture']
                          THE UNDERGROUND RAILROAD : A NOV

In [14]:
# For each matched book, check if the correct author exists in ebook_ranked for that month
results = []

for _, row in fuzzy_crossed_v3[fuzzy_crossed_v3['ebook_title'].notna()].head(50).iterrows():
    year  = row['checkoutyear']
    month = row['checkoutmonth']
    
    # Get all ebooks for that month
    ebook_month = ebook_ranked[(ebook_ranked['checkoutyear'] == year) &
                                (ebook_ranked['checkoutmonth'] == month)]
    
    # Check if correct author exists in ebook catalog that month
    author_score_max = ebook_month['creator'].apply(
        lambda a: fuzz.token_sort_ratio(row['book_author'], str(a))
    ).max()
    
    # Check if correct title exists in ebook catalog that month
    title_score_max = ebook_month['title'].apply(
        lambda t: fuzz.token_sort_ratio(row['book_title'], str(t))
    ).max()
    
    results.append({
        'book_title':         row['book_title'],
        'matched_ebook':      row['ebook_title'],
        'book_author':        row['book_author'],
        'matched_author':     row['ebook_author'],
        'title_score':        row['title_score'],
        'matched_author_score': fuzz.token_sort_ratio(row['book_author'], row['ebook_author']),
        'best_author_in_catalog': author_score_max,
        'best_title_in_catalog':  title_score_max,
    })

df_check = pd.DataFrame(results)

print("=== Title match returned wrong author? ===")
# Cases where the matched author is bad but a better author exists in catalog
wrong_author = df_check[
    (df_check['matched_author_score'] < 70) &
    (df_check['best_author_in_catalog'] >= 70)
]
print(f"Matched to wrong author but correct author exists: {len(wrong_author):,}")
print(wrong_author[['book_title', 'matched_ebook', 'book_author', 'matched_author',
                     'matched_author_score', 'best_author_in_catalog', 'best_title_in_catalog']
].to_string(index=False))

=== Title match returned wrong author? ===
Matched to wrong author but correct author exists: 13
                         book_title                     matched_ebook       book_author   matched_author  matched_author_score  best_author_in_catalog  best_title_in_catalog
THE WRONG SIDE OF GOODBYE : A NOVEL       THE ONE GOOD THING: A NOVEL CONNELLY, MICHAEL KEVIN ALAN MILNE             36.363636               96.969697              70.967742
THE WRONG SIDE OF GOODBYE : A NOVEL THE LOVE OF A GOOD WOMAN: STORIES CONNELLY, MICHAEL      ALICE MUNRO             28.571429               96.969697              73.529412
THE WRONG SIDE OF GOODBYE : A NOVEL THE LOVE OF A GOOD WOMAN: STORIES CONNELLY, MICHAEL      ALICE MUNRO             28.571429               96.969697              73.529412
                       NIGHT SCHOOL                        NIGHT SONG        CHILD, LEE  BEVERLY JENKINS             24.000000               94.736842              72.727273
                     INTO THE WAT

In [15]:
## Method 5 Fuzzy Match on title, break match ties on author, ADD validation criteria: 
### Check if authors match above threshold
### check if 3+ meaningful words from book title appears in ebook title

def validate_match(book_title, ebook_title, book_author, ebook_author,
                   author_threshold=70, min_substring_len=3):
    author_score = fuzz.token_sort_ratio(str(book_author), str(ebook_author))
    if author_score < author_threshold:
        return False, author_score, None
    
    book_words        = set(re.sub(r'[^a-z0-9\s]', ' ', str(book_title).lower()).split())
    ebook_title_lower = str(ebook_title).lower()
    noise             = {'a', 'an', 'the', 'of', 'in', 'and', 'or', 'to', 'by', 'for',
                         'novel', 'memoir', 'story', 'book', 'vol', 'volume'}
    meaningful_words  = [w for w in book_words if w not in noise and len(w) >= min_substring_len]
    matched_words     = [w for w in meaningful_words if w in ebook_title_lower]
    
    if matched_words:
        return True, author_score, matched_words
    return False, author_score, None


def fuzzy_cross_rank_title_author(book_df, ebook_df, book_rank_cutoff=5, tie_threshold=2,
                                   author_threshold=70, min_substring_len=3):
    results = []
    
    for (year, month), book_group in book_df[book_df['month_checkout_rank'] <= book_rank_cutoff].groupby(['checkoutyear', 'checkoutmonth']):
        ebook_month  = ebook_df[(ebook_df['checkoutyear'] == year) &
                                 (ebook_df['checkoutmonth'] == month)].drop_duplicates('title')
        ebook_titles  = ebook_month['title'].tolist()
        ebook_authors = ebook_month['creator'].tolist()

        for _, book_row in book_group.iterrows():
            title_candidates = process.extract(book_row['title'], ebook_titles,
                                               scorer=fuzz.token_sort_ratio, limit=5)
            
            if not title_candidates:
                best_match = best_title_score = best_author_score = best_matched_words = None
            else:
                top_score = title_candidates[0][1]
                tied      = [(title, score, idx) for title, score, idx in title_candidates
                             if top_score - score <= tie_threshold]
                
                best_match = best_title_score = best_author_score = best_matched_words = None

                if len(tied) == 1:
                    _, best_title_score, idx = tied[0]
                    candidate = ebook_month.iloc[idx]
                    is_valid, best_author_score, best_matched_words = validate_match(
                        book_row['title'], candidate['title'],
                        book_row['creator'], candidate['creator'],
                        author_threshold, min_substring_len
                    )
                    if is_valid:
                        best_match = candidate
                else:
                    best_combined_score = 0
                    for _, title_score, idx in tied:
                        candidate = ebook_month.iloc[idx]
                        is_valid, author_score, matched_words = validate_match(
                            book_row['title'], candidate['title'],
                            book_row['creator'], candidate['creator'],
                            author_threshold, min_substring_len
                        )
                        if not is_valid:
                            continue
                        combined_score = (title_score * 0.7) + (author_score * 0.3)
                        if combined_score > best_combined_score:
                            best_combined_score = combined_score
                            best_match          = candidate
                            best_title_score    = title_score
                            best_author_score   = author_score
                            best_matched_words  = matched_words

            if best_match is not None:
                results.append({
                    'checkoutyear':          year,
                    'checkoutmonth':         month,
                    'book_title':            book_row['title'],
                    'ebook_title':           best_match['title'],
                    'book_author':           book_row['creator'],
                    'ebook_author':          best_match['creator'],
                    'title_score':           best_title_score,
                    'author_score':          best_author_score,
                    'matched_words':         str(best_matched_words),
                    'genre':                 book_row['genre'],
                    'book_rank':             book_row['month_checkout_rank'],
                    'book_total_checkouts':  book_row['total_checkouts'],
                    'ebook_rank':            best_match['month_checkout_rank'],
                    'ebook_total_checkouts': best_match['total_checkouts'],
                })
            else:
                results.append({
                    'checkoutyear':          year,
                    'checkoutmonth':         month,
                    'book_title':            book_row['title'],
                    'ebook_title':           None,
                    'book_author':           book_row['creator'],
                    'ebook_author':          None,
                    'title_score':           None,
                    'author_score':          None,
                    'matched_words':         None,
                    'genre':                 book_row['genre'],
                    'book_rank':             book_row['month_checkout_rank'],
                    'book_total_checkouts':  book_row['total_checkouts'],
                    'ebook_rank':            None,
                    'ebook_total_checkouts': None,
                })
    
    return pd.DataFrame(results)

fuzzy_crossed_v4 = fuzzy_cross_rank_title_author(book_ranked, ebook_ranked, book_rank_cutoff=5)

total = len(fuzzy_crossed_v4)
matched = fuzzy_crossed_v4['ebook_rank'].notna().sum()

print("=== How often do top 5 physical books appear in ebook rankings with title + author tiebreak matching? ===")
print(f"Total book entries:    {total:,}")
print(f"Matched in ebooks:     {matched:,} ({matched/total*100:.1f}%)")
print(f"Not matched in ebooks: {total-matched:,} ({(total-matched)/total*100:.1f}%)")
print()

for tier in [5, 10, 20, 50]:
    count = (fuzzy_crossed_v4['ebook_rank'] <= tier).sum()
    print(f"In ebook top {tier:>2}: {count:>5,} ({count/total*100:.1f}%)")

=== How often do top 5 physical books appear in ebook rankings with title + author tiebreak matching? ===
Total book entries:    540
Matched in ebooks:     369 (68.3%)
Not matched in ebooks: 171 (31.7%)

In ebook top  5:    56 (10.4%)
In ebook top 10:    80 (14.8%)
In ebook top 20:   134 (24.8%)
In ebook top 50:   201 (37.2%)


In [16]:
# check unmatched results

unmatched = fuzzy_crossed_v4[fuzzy_crossed_v4['ebook_rank'].isna()].drop_duplicates('book_title')

candidates_list = []

for _, row in unmatched.iterrows():
    year  = row['checkoutyear']
    month = row['checkoutmonth']
    
    ebook_month   = ebook_ranked[(ebook_ranked['checkoutyear'] == year) &
                                  (ebook_ranked['checkoutmonth'] == month)].drop_duplicates('title')
    ebook_titles  = ebook_month['title'].tolist()
    ebook_authors = ebook_month['creator'].tolist()
    
    candidates = process.extract(row['book_title'], ebook_titles,
                                  scorer=fuzz.token_sort_ratio, limit=5)
    
    # Label each candidate by rank (1 = closest)
    for rank, (matched_title, title_score, idx) in enumerate(candidates, start=1):
        ebook_author = ebook_authors[idx]
        author_score = fuzz.token_sort_ratio(row['book_author'], ebook_author)
        combined     = (title_score * 0.7) + (author_score * 0.3)
        _, _, matched_words = validate_match(
            row['book_title'], matched_title,
            row['book_author'], ebook_author
        )
        candidates_list.append({
            'book_title':     row['book_title'],
            'book_author':    row['book_author'],
            'ebook_title':    matched_title,
            'ebook_author':   ebook_author,
            'ebook_candidate_rank': rank,
            'title_score':    round(title_score, 1),
            'author_score':   round(author_score, 1),
            'combined_score': round(combined, 1),
            'matched_words':  matched_words
        })

candidates_df = pd.DataFrame(candidates_list)

# Get top 20 book-ebook pairs by combined score
top20_books = (candidates_df.sort_values('combined_score', ascending=False)
               .drop_duplicates('book_title')
               .head(20)['book_title'].tolist())

# Print top 20 books with all 5 ebook candidates
print("=== Top 20 Unmatched Books by Best Combined Score — All 5 Ebook Candidates ===\n")
for book_title in top20_books:
    book_candidates = candidates_df[candidates_df['book_title'] == book_title].sort_values('ebook_candidate_rank')
    best = book_candidates.iloc[0]
    print(f"Book:   {best['book_title']}")
    print(f"Author: {best['book_author']}")
    print(f"  {'Rank':<6} {'Title Score':>11} {'Author Score':>12} {'Combined':>9} {'Matched Words':<20} Ebook Title | Ebook Author")
    for _, c in book_candidates.iterrows():
        print(f"  {c['ebook_candidate_rank']:<6} {c['title_score']:>11} {c['author_score']:>12} {c['combined_score']:>9} {str(c['matched_words']):<20} {c['ebook_title']} | {c['ebook_author']}")
    print()

=== Top 20 Unmatched Books by Best Combined Score — All 5 Ebook Candidates ===

Book:   THE FIFTH RISK
Author: LEWIS, MICHAEL (MICHAEL M.)
  Rank   Title Score Author Score  Combined Matched Words        Ebook Title | Ebook Author
  1            100.0         65.0      89.5 None                 THE FIFTH RISK | MICHAEL LEWIS
  2             80.0         25.6      63.7 None                 THE KING'S FIFTH | SCOTT O'DELL
  3             72.0         26.3      58.3 None                 THE LIFTERS | DAVE EGGERS
  4             69.2         30.0      57.5 None                 THE FIGHTERS | C. J. CHIVERS
  5             69.2         20.5      54.6 None                 THE GRIFTERS | JIM THOMPSON

Book:   GOING INFINITE : THE RISE AND FALL OF A NEW TYCOON
Author: LEWIS, MICHAEL (MICHAEL M.)
  Rank   Title Score Author Score  Combined Matched Words        Ebook Title | Ebook Author
  1             97.0         65.0      87.4 None                 GOING INFINITE: THE RISE AND FALL OF A NEW TY

In [138]:
## Method 6 
### Return closest fuzzy match author
### Return closest fuzzy match title that matches 75%+ of meaningful words from the book title

def fuzzy_cross_rank_v2(book_df, ebook_df, book_rank_cutoff=5,
                         author_threshold=70, min_substring_len=3):
    """
    Match on author first, then title.
    Return closest title match that also shares at least 1 meaningful word.
    """
    results = []

    #no change to matching performance if we also exclude common book club/ award names, so include to be safe
    noise = {'a', 'an', 'the', 'of', 'in', 'and', 'or', 'to', 'by', 'for',
         'novel', 'memoir', 'story', 'book', 'vol', 'volume', 'series',
         'pick', 'winner', 'prize', 'club', 'read', 'with', 'jenna',
         'oprah', 'reese', 'pulitzer', 'collection', 'continuing'}
    
    def clean_author(author):
        author = re.sub(r'\(.*?\)', '', str(author))
        author = re.sub(r',?\s*(JR|SR|II|III|IV)\.?', '', author, flags=re.IGNORECASE)
        author = re.sub(r',?\s*\d{4}-?(\d{4})?', '', author)
        return author.strip()

    def get_meaningful_words(title, author=None):
        words = set(re.sub(r'[^a-z0-9\s]', ' ', str(title).lower()).split())
        author_words = set(re.sub(r'[^a-z0-9\s]', ' ', str(author).lower()).split()) if author else set()
        return [w for w in words 
                if w not in noise
                and len(w) >= min_substring_len
                and w not in author_words]

    for (year, month), book_group in book_df[book_df['month_checkout_rank'] <= book_rank_cutoff].groupby(['checkoutyear', 'checkoutmonth']):
        ebook_month   = ebook_df[(ebook_df['checkoutyear'] == year) &
                                  (ebook_df['checkoutmonth'] == month)].drop_duplicates('title')
        ebook_authors = ebook_month['creator'].tolist()

        for _, book_row in book_group.iterrows():
            book_author_clean = clean_author(book_row['creator'])

            # Step 1 — fuzzy match on author, filter above threshold
            author_candidates = process.extract(book_author_clean, ebook_authors,
                                                 scorer=fuzz.token_sort_ratio, limit=10)
            passing_authors = [(score, idx) for _, score, idx in author_candidates
                               if score >= author_threshold]

            if not passing_authors:
                best_match = best_title_score = best_author_score = best_matched_words = None
            else:
                # Step 2 — among passing authors, score each title using WRatio
                # WRatio handles short vs long title comparisons better than token_sort_ratio
                title_candidates = []
                book_words = get_meaningful_words(book_row['title'], book_row['creator'])
                min_words_required = max(1, len(book_words) * .75)  # match at least 75% of meaningful words

                for author_score, idx in passing_authors:
                    candidate     = ebook_month.iloc[idx]
                    title_score   = fuzz.WRatio(book_row['title'], candidate['title'])
                    ebook_lower   = candidate['title'].lower()
                    matched_words = [w for w in book_words if w in ebook_lower]

                    # Only consider candidates matching >= half # meaningful words in book title
                    if len(matched_words) >= min_words_required:
                        title_candidates.append((title_score, author_score, idx, matched_words))

                if not title_candidates:
                    best_match = best_title_score = best_author_score = best_matched_words = None
                else:
                    # Take the candidate with the highest title score
                    title_candidates.sort(key=lambda x: x[0], reverse=True)
                    best_title_score, best_author_score, idx, best_matched_words = title_candidates[0]
                    best_match = ebook_month.iloc[idx]

            if best_match is not None:
                results.append({
                    'checkoutyear':          year,
                    'checkoutmonth':         month,
                    'month_date':            book_row['month_date'],
                    'book_title':            book_row['title'],
                    'ebook_title':           best_match['title'],
                    'book_author':           book_row['creator'],
                    'ebook_author':          best_match['creator'],
                    'title_score':           best_title_score,
                    'author_score':          best_author_score,
                    'matched_words':         str(best_matched_words),
                    'genre':                 book_row['genre'],
                    'book_rank':             book_row['month_checkout_rank'],
                    'book_total_checkouts':  book_row['total_checkouts'],
                    'ebook_rank':            best_match['month_checkout_rank'],
                    'ebook_total_checkouts': best_match['total_checkouts'],
                })
            else:
                results.append({
                    'checkoutyear':          year,
                    'checkoutmonth':         month,
                    'month_date':            book_row['month_date'],
                    'book_title':            book_row['title'],
                    'ebook_title':           None,
                    'book_author':           book_row['creator'],
                    'ebook_author':          None,
                    'title_score':           None,
                    'author_score':          None,
                    'matched_words':         None,
                    'genre':                 book_row['genre'],
                    'book_rank':             book_row['month_checkout_rank'],
                    'book_total_checkouts':  book_row['total_checkouts'],
                    'ebook_rank':            None,
                    'ebook_total_checkouts': None,
                })

    return pd.DataFrame(results)

fuzzy_crossed_v6 = fuzzy_cross_rank_v2(book_ranked, ebook_ranked, book_rank_cutoff=5)

total   = len(fuzzy_crossed_v6)
matched = fuzzy_crossed_v6['ebook_rank'].notna().sum()

print("=== Method 6: Fuzzy author match → closest title with 1+ meaningful word ===")
print(f"Total book entries:    {total:,}")
print(f"Matched in ebooks:     {matched:,} ({matched/total*100:.1f}%)")
print(f"Not matched in ebooks: {total-matched:,} ({(total-matched)/total*100:.1f}%)")
print()
for tier in [5, 10, 20, 50]:
    count = (fuzzy_crossed_v6['ebook_rank'] <= tier).sum()
    print(f"In ebook top {tier:>2}: {count:>5,} ({count/total*100:.1f}%)")

=== Method 6: Fuzzy author match → closest title with 1+ meaningful word ===
Total book entries:    540
Matched in ebooks:     493 (91.3%)
Not matched in ebooks: 47 (8.7%)

In ebook top  5:    85 (15.7%)
In ebook top 10:   116 (21.5%)
In ebook top 20:   182 (33.7%)
In ebook top 50:   275 (50.9%)


In [18]:
##check lowest similarity score matches to confirm they're correct
matched = fuzzy_crossed_v6[fuzzy_crossed_v6['ebook_rank'].notna()].copy()
matched['num_matched_words'] = matched['matched_words'].apply(
    lambda x: len(eval(x)) if x is not None and x != 'None' else 0
)

# Weight: author most important, then title, then word count
matched['combined_score'] = (
    (matched['author_score'] * 0.5) +
    (matched['title_score']  * 0.4) +
    (matched['num_matched_words'] * 2)  # small boost per word
)

worst = (matched
         .drop_duplicates('book_title')
         .nsmallest(20, 'combined_score'))

print("=== 20 Lowest Combined Score Matches ===")
for _, row in worst.iterrows():
    print(f"{'─' * 80}")
    print(f"Book:          {row['book_title'][:75]}")
    print(f"Ebook:         {row['ebook_title'][:75]}")
    print(f"Book Author:   {row['book_author'][:50]}")
    print(f"Ebook Author:  {row['ebook_author'][:50]}")
    print(f"Title Score:   {row['title_score']:.1f}  |  Author Score: {row['author_score']:.1f}  |  Matched Words: {row['num_matched_words']}")
    print(f"Words Matched: {row['matched_words']}")
    print(f"Combined Score: {row['combined_score']}")
print(f"{'─' * 80}")

=== 20 Lowest Combined Score Matches ===
────────────────────────────────────────────────────────────────────────────────
Book:          HELLO BEAUTIFUL : A NOVEL
Ebook:         HELLO BEAUTIFUL: OPRAH'S BOOK CLUB
Book Author:   NAPOLITANO, ANN
Ebook Author:  ANN NAPOLITANO
Title Score:   71.2  |  Author Score: 96.6  |  Matched Words: 2
Words Matched: ['beautiful', 'hello']
Combined Score: 80.75043834015196
────────────────────────────────────────────────────────────────────────────────
Book:          THE LIFE IMPOSSIBLE
Ebook:         THE LIFE IMPOSSIBLE: A NOVEL
Book Author:   HAIG, MATT
Ebook Author:  MATT HAIG
Title Score:   80.9  |  Author Score: 94.7  |  Matched Words: 2
Words Matched: ['life', 'impossible']
Combined Score: 83.70884658454648
────────────────────────────────────────────────────────────────────────────────
Book:          TRANSCENDENT KINGDOM
Ebook:         TRANSCENDENT KINGDOM: A NOVEL
Book Author:   GYASI, YAA
Ebook Author:  YAA GYASI
Title Score:   81.6  |  Author

In [19]:
### check unmatched results 
unmatched = fuzzy_crossed_v6[fuzzy_crossed_v6['ebook_rank'].isna()].drop_duplicates('book_title')

candidates_list = []

#reimport clean_author and get_meaningful_words functions

min_substring_len=3
noise = {'a', 'an', 'the', 'of', 'in', 'and', 'or', 'to', 'by', 'for',
             'novel', 'memoir', 'story', 'book', 'vol', 'volume'}

def clean_author(author):
        author = re.sub(r'\(.*?\)', '', str(author))
        author = re.sub(r',?\s*(JR|SR|II|III|IV)\.?', '', author, flags=re.IGNORECASE)
        author = re.sub(r',?\s*\d{4}-?(\d{4})?', '', author)
        return author.strip()

def get_meaningful_words(title, author=None):
        words = set(re.sub(r'[^a-z0-9\s]', ' ', str(title).lower()).split())
        author_words = set(re.sub(r'[^a-z0-9\s]', ' ', str(author).lower()).split()) if author else set()
        return [w for w in words 
                if w not in noise
                and len(w) >= min_substring_len
                and w not in author_words]

for _, row in unmatched.iterrows():
    year  = row['checkoutyear']
    month = row['checkoutmonth']
    
    ebook_month   = ebook_ranked[(ebook_ranked['checkoutyear'] == year) &
                                  (ebook_ranked['checkoutmonth'] == month)].drop_duplicates('title')
    ebook_authors = ebook_month['creator'].tolist()
    
    # Clean author for matching — same logic as in fuzzy_cross_rank_v2
    book_author_clean = clean_author(row['book_author'])
    book_words        = get_meaningful_words(row['book_title'], row['book_author'])
    min_words_required = max(1, len(book_words) * 0.75)

    # Step 1 — get passing authors
    author_candidates = process.extract(book_author_clean, ebook_authors,
                                         scorer=fuzz.token_sort_ratio, limit=10)
    passing_authors = [(score, idx) for _, score, idx in author_candidates
                       if score >= 70]

    # Step 2 — score titles among passing authors
    title_candidates = []
    for author_score, idx in passing_authors:
        candidate     = ebook_month.iloc[idx]
        title_score   = fuzz.WRatio(row['book_title'], candidate['title'])
        ebook_lower   = candidate['title'].lower()
        matched_words = [w for w in book_words if w in ebook_lower]
        combined      = (title_score * 0.6) + (author_score * 0.4)
        title_candidates.append({
            'book_title':           row['book_title'],
            'book_author':          row['book_author'],
            'ebook_title':          candidate['title'],
            'ebook_author':         candidate['creator'],
            'title_score':          round(title_score, 1),
            'author_score':         round(author_score, 1),
            'combined_score':       round(combined, 1),
            'matched_words':        matched_words,
            'num_matched_words':    len(matched_words),
            'min_words_required':   min_words_required,
            'passed_word_filter':   len(matched_words) >= min_words_required,
        })

    # Sort by combined score descending
    title_candidates.sort(key=lambda x: x['combined_score'], reverse=True)
    
    # Take top 5
    for rank, c in enumerate(title_candidates[:5], start=1):
        c['ebook_candidate_rank'] = rank
        candidates_list.append(c)

candidates_df = pd.DataFrame(candidates_list)

# Get top 20 unmatched books by best combined score
top20_books = (candidates_df.sort_values('combined_score', ascending=False)
               .drop_duplicates('book_title')
               .head(20)['book_title'].tolist())

print("=== Top 20 Unmatched Books (Method 6) — Top 5 Author-Matched Ebook Candidates ===\n")
for book_title in top20_books:
    book_candidates = candidates_df[candidates_df['book_title'] == book_title].sort_values('ebook_candidate_rank')
    best = book_candidates.iloc[0]
    min_req = best['min_words_required']
    print(f"Book:    {best['book_title'][:75]}")
    print(f"Author:  {best['book_author'][:50]}")
    print(f"Meaningful words: {get_meaningful_words(best['book_title'], best['book_author'])} → Min required: {min_req:.1f}")
    print(f"  {'Rank':<5} {'Title':>5} {'Author':>7} {'Combined':>9} {'Words':>6} {'Pass':>5}  Ebook Title | Ebook Author")
    for _, c in book_candidates.iterrows():
        passed = '✓' if c['passed_word_filter'] else '✗'
        print(f"  {c['ebook_candidate_rank']:<5} {c['title_score']:>5} {c['author_score']:>7} {c['combined_score']:>9} {c['num_matched_words']:>3}/{min_req:<3.0f} {passed:>5}  {c['ebook_title'][:50]} | {c['ebook_author'][:30]}")
    print()

=== Top 20 Unmatched Books (Method 6) — Top 5 Author-Matched Ebook Candidates ===

Book:    ATMOSPHERE : A LOVE STORY
Author:  REID, TAYLOR JENKINS
Meaningful words: ['love', 'atmosphere'] → Min required: 1.5
  Rank  Title  Author  Combined  Words  Pass  Ebook Title | Ebook Author
  1      90.0    97.4      93.0   1/2       ✗  ATMOSPHERE | TAYLOR JENKINS REID
  2      85.5    97.4      90.3   0/2       ✗  THE SEVEN HUSBANDS OF EVELYN HUGO: A NOVEL | TAYLOR JENKINS REID
  3      85.5    97.4      90.3   0/2       ✗  MALIBU RISING: A READ WITH JENNA PICK: A NOVEL | TAYLOR JENKINS REID
  4      54.5    97.4      71.7   0/2       ✗  AFTER I DO: A NOVEL | TAYLOR JENKINS REID
  5      50.0    97.4      69.0   1/2       ✗  ONE TRUE LOVES: A NOVEL | TAYLOR JENKINS REID

Book:    THE LET THEM THEORY : A LIFE-CHANGING TOOL THAT MILLIONS OF PEOPLE CAN'T ST
Author:  ROBBINS, MEL
Meaningful words: ['them', 'changing', 'life', 'can', 'stop', 'let', 'tool', 'millions', 'theory', 'talking', 'about', '

## Match Top Ebooks Book Titles to Popular Physical Book Titles, by Monthly Checkout rank

In [87]:
## match on naive approach
## fuzzy match authors, normalize titles (lowercase, strip punctuation, extra spaces), 
## check if the physical book title is a substring of the ebook title


def substring_cross_rank(ebook_df, book_df, ebook_rank_cutoff=10,
                         book_rank_cutoff=50, author_threshold=70):
    """
    Match ebooks to physical books in two steps:
      1. Fuzzy match on author (token_sort_ratio >= threshold)
      2. Among fuzzy-matched authors, check if physical book title is a substring of ebook title
    Also records best author + title scores for ALL candidates, matched or not.
    """

    def normalize(text):
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def clean_author(author):
        author = re.sub(r'\(.*?\)', '', str(author))
        author = re.sub(r',?\s*(JR|SR|II|III|IV)\.?', '', author, flags=re.IGNORECASE)
        author = re.sub(r',?\s*\d{4}-?(\d{4})?', '', author)
        return author.strip()

    results = []

    for (year, month), ebook_group in (
        ebook_df[ebook_df['month_checkout_rank'] <= ebook_rank_cutoff]
        .groupby(['checkoutyear', 'checkoutmonth'])
    ):
        book_month = (
            book_df[
                (book_df['checkoutyear'] == year) &
                (book_df['checkoutmonth'] == month) &
                (book_df['month_checkout_rank'] <= book_rank_cutoff)
            ]
            .drop_duplicates('title')
            .copy()
        )
        book_month['title_norm'] = book_month['title'].apply(normalize)
        book_authors = book_month['creator'].tolist()

        for _, ebook_row in ebook_group.iterrows():
            ebook_norm         = normalize(ebook_row['title'])
            ebook_author_clean = clean_author(ebook_row['creator'])

            # Step 1 — fuzzy author match
            author_candidates = process.extract(ebook_author_clean, book_authors,
                                                scorer=fuzz.token_sort_ratio, limit=10)
            passing_authors = [(score, idx) for _, score, idx in author_candidates
                               if score >= author_threshold]

            # track best author + title score across ALL candidates regardless of match
            best_overall_author_score = max((s for s, _ in [(score, idx) for _, score, idx in author_candidates]), default=None)
            best_overall_title_score  = None
            if author_candidates:
                for _, author_score, idx in author_candidates:
                    candidate   = book_month.iloc[idx]
                    title_score = fuzz.WRatio(ebook_row['title'], candidate['title'])
                    if best_overall_title_score is None or title_score > best_overall_title_score:
                        best_overall_title_score = title_score

            # Step 2 — among passing authors, check if physical title is substring of ebook title
            matches = []
            for author_score, idx in passing_authors:
                candidate = book_month.iloc[idx]
                if candidate['title_norm'] in ebook_norm:
                    matches.append((author_score, idx))

            if matches:
                matches.sort(key=lambda x: (-x[0], book_month.iloc[x[1]]['month_checkout_rank']))
                best_author_score, best_idx = matches[0]
                best = book_month.iloc[best_idx]
                best_title_score = fuzz.WRatio(ebook_row['title'], best['title'])
                results.append({
                    'checkoutyear':               year,
                    'checkoutmonth':              month,
                    'month_date':                 ebook_row.get('month_date'),
                    'ebook_title':                ebook_row['title'],
                    'book_title':                 best['title'],
                    'ebook_author':               ebook_row['creator'],
                    'book_author':                best['creator'],
                    'ebook_norm':                 ebook_norm,
                    'book_norm':                  best['title_norm'],
                    'author_score':               best_author_score,       # score of matched author
                    'title_score':                best_title_score,        # fuzzy score of matched title
                    'best_overall_author_score':  best_overall_author_score, # best author score seen
                    'best_overall_title_score':   best_overall_title_score,  # best title score seen
                    'match_type':                 'author+title',
                    'genre':                      ebook_row.get('genre'),
                    'ebook_rank':                 ebook_row['month_checkout_rank'],
                    'ebook_total_checkouts':      ebook_row.get('total_checkouts'),
                    'book_rank':                  best['month_checkout_rank'],
                    'book_total_checkouts':       best.get('total_checkouts'),
                })
            else:
                results.append({
                    'checkoutyear':               year,
                    'checkoutmonth':              month,
                    'month_date':                 ebook_row.get('month_date'),
                    'ebook_title':                ebook_row['title'],
                    'book_title':                 None,
                    'ebook_author':               ebook_row['creator'],
                    'book_author':                None,
                    'ebook_norm':                 ebook_norm,
                    'book_norm':                  None,
                    'author_score':               None,
                    'title_score':                None,
                    'best_overall_author_score':  best_overall_author_score,
                    'best_overall_title_score':   best_overall_title_score,
                    'match_type':                 'author_only' if passing_authors else 'no_match',
                    'genre':                      ebook_row.get('genre'),
                    'ebook_rank':                 ebook_row['month_checkout_rank'],
                    'ebook_total_checkouts':      ebook_row.get('total_checkouts'),
                    'book_rank':                  None,
                    'book_total_checkouts':       None,
                })

    return pd.DataFrame(results)


substring_matched = substring_cross_rank(ebook_ranked, book_ranked,
                                         ebook_rank_cutoff=10,
                                         book_rank_cutoff=50,
                                         author_threshold=70)

## Performance metrics
total   = len(substring_matched)
matched = substring_matched['book_rank'].notna().sum()

print("=== Substring Match: physical title ⊆ ebook title ===")
print(f"Total ebook entries         : {total:,}")
print(f"Matched to physical book    : {matched:,} ({matched/total*100:.1f}%)")
print(f"Not matched                 : {total-matched:,} ({(total-matched)/total*100:.1f}%)")
print()

print("── Failure breakdown ──")
fail_counts = substring_matched[substring_matched['book_rank'].isna()]['match_type'].value_counts()
for mtype, count in fail_counts.items():
    print(f"  {mtype:<15}: {count:>5,} ({count/total*100:.1f}%)")
print()

print("── Match rate by physical book rank tier ──")
for tier in [5, 10, 20, 50]:
    count = (substring_matched['book_rank'] <= tier).sum()
    print(f"  Physical book top {tier:>2}: {count:>5,} ({count/total*100:.1f}%)")

print("\n── Sample MATCHED pairs ──")
print(
    substring_matched[substring_matched['book_rank'].notna()]
    [['ebook_title', 'book_title', 'ebook_rank', 'book_rank', 'author_score']]
    .drop_duplicates('ebook_title')
    .head(5)
    .to_string()
)

print("\n── Sample NOT matched ebook titles ──")
print(
    substring_matched[substring_matched['book_rank'].isna()]
    [['ebook_title', 'ebook_author', 'ebook_rank', 'match_type']]
    .drop_duplicates('ebook_title')
    .head(5)
    .to_string()
)

=== Substring Match: physical title ⊆ ebook title ===
Total ebook entries         : 1,080
Matched to physical book    : 274 (25.4%)
Not matched                 : 806 (74.6%)

── Failure breakdown ──
  no_match       :   719 (66.6%)
  author_only    :    87 (8.1%)

── Match rate by physical book rank tier ──
  Physical book top  5:    96 (8.9%)
  Physical book top 10:   122 (11.3%)
  Physical book top 20:   165 (15.3%)
  Physical book top 50:   274 (25.4%)

── Sample MATCHED pairs ──
                                                                               ebook_title                                                                               book_title  ebook_rank  book_rank  author_score
4                                                                 BETWEEN THE WORLD AND ME                                                                 BETWEEN THE WORLD AND ME           5        8.0     96.969697
6   THE LIFE-CHANGING MAGIC OF TIDYING UP: THE JAPANESE ART OF DECLUTTERING AN

In [88]:
# ── Debug: for each unmatched ebook, show physical book candidates considered ──
def debug_unmatched(ebook_df, book_df, ebook_rank_cutoff=10, book_rank_cutoff=50):

    def normalize(text):
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    debug_rows = []

    for (year, month), ebook_group in (
        ebook_df[ebook_df['month_checkout_rank'] <= ebook_rank_cutoff]
        .groupby(['checkoutyear', 'checkoutmonth'])
    ):
        book_month = (
            book_df[
                (book_df['checkoutyear'] == year) &
                (book_df['checkoutmonth'] == month) &
                (book_df['month_checkout_rank'] <= book_rank_cutoff)
            ]
            .drop_duplicates('title')
            .copy()
        )
        book_month['title_norm'] = book_month['title'].apply(normalize)

        for _, ebook_row in ebook_group.iterrows():
            ebook_norm = normalize(ebook_row['title'])

            # check if any physical book title is a substring of this ebook title
            mask = book_month['title_norm'].apply(
                lambda t: t in ebook_norm
            )

            # only process unmatched ebooks
            if mask.any():
                continue

            for _, book_row in book_month.iterrows():
                debug_rows.append({
                    'checkoutyear':  year,
                    'checkoutmonth': month,
                    'ebook_title':   ebook_row['title'],
                    'ebook_norm':    ebook_norm,
                    'ebook_author':  ebook_row['creator'],
                    'ebook_rank':    ebook_row['month_checkout_rank'],
                    'book_title':    book_row['title'],
                    'book_norm':     book_row['title_norm'],
                    'book_rank':     book_row['month_checkout_rank'],
                    'book_author':   book_row['creator'],
                })

    return pd.DataFrame(debug_rows)


debug_df = debug_unmatched(ebook_ranked, book_ranked,
                           ebook_rank_cutoff=10,
                           book_rank_cutoff=50)

print("── Breakdown of unmatched ebook titles by failure reason ──")
print(
    substring_matched[substring_matched['book_rank'].isna()]
    .groupby('match_type')
    .agg(distinct_ebooks=('ebook_norm', 'nunique'))
    .reset_index()
)

print("\n── Sample 5 ebook titles where author fuzzy match failed ──")
print(
    substring_matched[
        (substring_matched['book_rank'].isna()) &
        (substring_matched['match_type'] == 'no_match')
    ][['ebook_title', 'ebook_author', 'ebook_norm', 'ebook_rank']]
    .drop_duplicates('ebook_title')
    .head(5)
    .to_string()
)

print("\n── Sample 5 ebook titles where author matched but title substring failed ──")
print(
    substring_matched[
        (substring_matched['book_rank'].isna()) &
        (substring_matched['match_type'] == 'author_only')
    ][['ebook_title', 'ebook_author', 'ebook_norm', 'ebook_rank']]
    .drop_duplicates('ebook_title')
    .head(5)
    .to_string()
)

── Breakdown of unmatched ebook titles by failure reason ──
    match_type  distinct_ebooks
0  author_only               32
1     no_match              155

── Sample 5 ebook titles where author fuzzy match failed ──
                                                                                                           ebook_title   ebook_author                                                                                                        ebook_norm  ebook_rank
0                                                                  THE GOLDFINCH: A NOVEL (PULITZER PRIZE FOR FICTION)    DONNA TARTT                                                                  the goldfinch a novel pulitzer prize for fiction           1
5                                                                                                 THE MARTIAN: A NOVEL      ANDY WEIR                                                                                               the martian a novel           6
7  

In [112]:
# Top 10 unmatched ebooks with the best-scoring physical book candidate ──
def get_best_candidate(ebook_df, book_df, ebook_rank_cutoff=10,
                       book_rank_cutoff=50, author_threshold=70):
    """
    For each ebook that failed to match in a given month, return the best
    physical book candidate from that same month.
    """
    def normalize(text):
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def clean_author(author):
        author = re.sub(r'\(.*?\)', '', str(author))
        author = re.sub(r',?\s*(JR|SR|II|III|IV)\.?', '', author, flags=re.IGNORECASE)
        author = re.sub(r',?\s*\d{4}-?(\d{4})?', '', author)
        return author.strip()

    # unmatched ebook+month combinations — one row per ebook per month
    unmatched_month = (
        substring_matched[substring_matched['book_rank'].isna()]
        [['ebook_title', 'checkoutyear', 'checkoutmonth', 'match_type']]
        .drop_duplicates()
    )

    rows = []

    for (year, month), ebook_group in (
        ebook_df[ebook_df['month_checkout_rank'] <= ebook_rank_cutoff]
        .groupby(['checkoutyear', 'checkoutmonth'])
    ):
        # only process months that have unmatched ebooks
        month_unmatched = unmatched_month[
            (unmatched_month['checkoutyear']  == year) &
            (unmatched_month['checkoutmonth'] == month)
        ]
        if month_unmatched.empty:
            continue

        unmatched_this_month = set(month_unmatched['ebook_title'])

        book_month = (
            book_df[
                (book_df['checkoutyear'] == year) &
                (book_df['checkoutmonth'] == month) &
                (book_df['month_checkout_rank'] <= book_rank_cutoff)
            ]
            .drop_duplicates('title')
            .copy()
        )
        book_month['title_norm'] = book_month['title'].apply(normalize)
        book_authors = book_month['creator'].tolist()

        for _, ebook_row in ebook_group.iterrows():
            if ebook_row['title'] not in unmatched_this_month:
                continue

            ebook_norm         = normalize(ebook_row['title'])
            ebook_author_clean = clean_author(ebook_row['creator'])

            author_candidates = process.extract(ebook_author_clean, book_authors,
                                                scorer=fuzz.token_sort_ratio, limit=10)

            scored = []
            for _, author_score, idx in author_candidates:
                candidate   = book_month.iloc[idx]
                title_score = fuzz.WRatio(ebook_row['title'], candidate['title'])
                scored.append((author_score, title_score,
                               candidate['title'], candidate['creator']))

            if not scored:
                continue

            scored.sort(key=lambda x: (-x[0], -x[1]))
            best_author_score, best_title_score, best_book_title, best_book_author = scored[0]

            rows.append({
                'checkoutyear':      year,
                'checkoutmonth':     month,
                'ebook_title':       ebook_row['title'],
                'ebook_author':      ebook_row['creator'],
                'ebook_norm':        ebook_norm,
                'ebook_rank':        ebook_row['month_checkout_rank'],
                'best_book_title':   best_book_title,
                'best_book_author':  best_book_author,
                'best_author_score': best_author_score,
                'best_title_score':  best_title_score,
                'match_type':        month_unmatched[
                                       month_unmatched['ebook_title'] == ebook_row['title']
                                     ]['match_type'].iloc[0],
            })

    return (
        pd.DataFrame(rows)
        .sort_values(['best_author_score', 'best_title_score'], ascending=False)
        .reset_index(drop=True)
    )


best_candidates = get_best_candidate(ebook_ranked, book_ranked,
                                     ebook_rank_cutoff=10,
                                     book_rank_cutoff=50)

print(f"Total unmatched ebook-month rows : {len(best_candidates):,}")
print(f"Distinct ebook titles            : {best_candidates['ebook_title'].nunique():,}")
print()

print("=== Top 20 unmatched ebook-months — best physical book candidate that month ===")
unmatched = (
    best_candidates
    [['checkoutyear', 'checkoutmonth', 'ebook_title', 'ebook_author',
      'best_book_title', 'best_book_author',
      'best_author_score', 'best_title_score', 'match_type', 'ebook_rank']]
    )

unmatched.head(20)

Total unmatched ebook-month rows : 806
Distinct ebook titles            : 165

=== Top 20 unmatched ebook-months — best physical book candidate that month ===


,checkoutyear,checkoutmonth,ebook_title,ebook_author,best_book_title,best_book_author,best_author_score,best_title_score,match_type,ebook_rank
0,2025,9,ATMOSPHERE,TAYLOR JENKINS REID,ATMOSPHERE : A LOVE STORY,"REID, TAYLOR JENKINS",97.435897,90.000000,author_only,10
1,2025,12,ATMOSPHERE,TAYLOR JENKINS REID,ATMOSPHERE : A LOVE STORY,"REID, TAYLOR JENKINS",97.435897,90.000000,author_only,5
2,2022,9,THE SEVEN HUSBANDS OF EVELYN HUGO: A NOVEL,TAYLOR JENKINS REID,CARRIE SOTO IS BACK : A NOVEL,"REID, TAYLOR JENKINS",97.435897,47.887324,author_only,8
3,2022,11,THE SEVEN HUSBANDS OF EVELYN HUGO: A NOVEL,TAYLOR JENKINS REID,CARRIE SOTO IS BACK : A NOVEL,"REID, TAYLOR JENKINS",97.435897,47.887324,author_only,7
4,2022,12,THE SEVEN HUSBANDS OF EVELYN HUGO: A NOVEL,TAYLOR JENKINS REID,CARRIE SOTO IS BACK : A NOVEL,"REID, TAYLOR JENKINS",97.435897,47.887324,author_only,7
5,2022,9,THE HOUSE OF BROKEN ANGELS,LUIS ALBERTO URREA,THE HOUSE OF BROKEN ANGELS : A NOVEL,"URREA, LUIS ALBERTO",97.297297,95.000000,author_only,1
6,2022,10,THE HOUSE OF BROKEN ANGELS,LUIS ALBERTO URREA,THE HOUSE OF BROKEN ANGELS : A NOVEL,"URREA, LUIS ALBERTO",97.297297,95.000000,author_only,1
7,2024,3,DEMON COPPERHEAD,BARBARA KINGSOLVER,DEMON COPPERHEAD : A NOVEL,"KINGSOLVER, BARBARA",97.297297,90.000000,author_only,10
8,2024,4,DEMON COPPERHEAD,BARBARA KINGSOLVER,DEMON COPPERHEAD : A NOVEL,"KINGSOLVER, BARBARA",97.297297,90.000000,author_only,2
9,2024,12,DEMON COPPERHEAD,BARBARA KINGSOLVER,DEMON COPPERHEAD : A NOVEL,"KINGSOLVER, BARBARA",97.297297,90.000000,author_only,2


In [113]:
# bottom 10 matched ebooks by lowest scores (weakest matches) ──
print("\n\n === Bottom 20 matched ebook-months by lowest author+title scores ===")

matched = (
    substring_matched[substring_matched['book_rank'].notna()]
    .sort_values(['author_score', 'title_score'])
    [['checkoutyear', 'checkoutmonth', 'ebook_title', 'book_title',
      'ebook_author', 'book_author',
      'author_score', 'title_score', 'ebook_rank', 'book_rank']]
    .reset_index(drop=True)
)

matched.head(20)



 === Bottom 20 matched ebook-months by lowest author+title scores ===


,checkoutyear,checkoutmonth,ebook_title,book_title,ebook_author,book_author,author_score,title_score,ebook_rank,book_rank
0,2017,1,THE LIFE-CHANGING MAGIC OF TIDYING UP: THE JAP...,THE LIFE-CHANGING MAGIC OF TIDYING UP : THE JA...,MARIE KONDO,"KONDŌ, MARIE",86.956522,99.421965,7,32.0
1,2017,2,THE LIFE-CHANGING MAGIC OF TIDYING UP: THE JAP...,THE LIFE-CHANGING MAGIC OF TIDYING UP : THE JA...,MARIE KONDO,"KONDŌ, MARIE",86.956522,99.421965,9,25.0
2,2025,2,JAMES: A NOVEL,JAMES : A NOVEL,PERCIVAL EVERETT,"EVERETT, PERCIVAL",90.322581,96.551724,8,12.0
3,2025,3,JAMES: A NOVEL,JAMES : A NOVEL,PERCIVAL EVERETT,"EVERETT, PERCIVAL",90.322581,96.551724,6,1.0
4,2017,1,THE GIRLS: A NOVEL,THE GIRLS : A NOVEL,EMMA CLINE,"CLINE, EMMA.",90.909091,97.297297,9,12.0
5,2023,1,SPARE,SPARE,"PRINCE HARRY, THE DUKE OF SUSSEX","HARRY, PRINCE, DUKE OF SUSSEX",91.803279,100.000000,6,1.0
6,2023,2,SPARE,SPARE,"PRINCE HARRY, THE DUKE OF SUSSEX","HARRY, PRINCE, DUKE OF SUSSEX",91.803279,100.000000,7,1.0
7,2023,3,SPARE,SPARE,"PRINCE HARRY, THE DUKE OF SUSSEX","HARRY, PRINCE, DUKE OF SUSSEX",91.803279,100.000000,8,1.0
8,2023,4,SPARE,SPARE,"PRINCE HARRY, THE DUKE OF SUSSEX","HARRY, PRINCE, DUKE OF SUSSEX",91.803279,100.000000,9,2.0
9,2021,4,THE MIDNIGHT LIBRARY: A NOVEL,THE MIDNIGHT LIBRARY,MATT HAIG,"HAIG, MATT",94.736842,81.632653,3,46.0


In [135]:
## pivot to checking the substrings that aren't shared between the closet fuzzy matched titles

def get_title_diff(ebook_title, book_title):
    """Return the part of the ebook title that isn't in the book title."""
    ebook_lower = str(ebook_title).lower()
    book_lower  = str(book_title).lower()
    
    idx = ebook_lower.find(book_lower)
    
    if idx == -1:
        return ebook_title  # no overlap found — return full ebook title
    
    before = ebook_title[:idx].strip(' :-')
    after  = ebook_title[idx + len(book_title):].strip(' :-')
    
    return ' | '.join(filter(None, [before, after]))


unmatched = (
    best_candidates
    [['checkoutyear', 'checkoutmonth', 'ebook_title', 'ebook_author',
      'best_book_title', 'best_book_author',
      'best_author_score', 'best_title_score', 'match_type', 'ebook_rank']]
    .copy()
)

unmatched['title_diff'] = unmatched.apply(
    lambda row: get_title_diff(row['ebook_title'], row['best_book_title'])
    if pd.notna(row['best_book_title']) else None,
    axis=1
)

# explode on | separator so each diff fragment gets its own row
diff_counts = (
    unmatched[unmatched['title_diff'].notna() & (unmatched['title_diff'] != '')]
    .assign(title_diff=lambda df: df['title_diff'].str.split(' | '))
    .explode('title_diff')
    .groupby('title_diff')
    .agg(
        distinct_ebook_titles = ('ebook_title', 'nunique'),
        occurrences           = ('ebook_title', 'count'),
    )
    .sort_values('distinct_ebook_titles', ascending=False)
    .reset_index()
)

# add example ebook titles -> physical book titles for each diff

diff_with_examples = (
    unmatched[unmatched['title_diff'].notna() & (unmatched['title_diff'] != '')]
    .assign(title_diff=lambda df: df['title_diff'].str.split(' | '))
    .explode('title_diff')
)

examples = (
    diff_with_examples
    .groupby('title_diff')
    .apply(lambda x: list(
        x[['ebook_title', 'best_book_title']]
        .drop_duplicates('ebook_title')
        .head(5)
        .itertuples(index=False, name=None)
    ))
    .reset_index()
    .rename(columns={0: 'examples'})
)

diff_counts = (
    diff_with_examples
    .groupby('title_diff')
    .agg(
        distinct_ebook_titles = ('ebook_title', 'nunique'),
        occurrences           = ('ebook_title', 'count'),
    )
    .reset_index()
)

diff_summary = (
    diff_counts
    .merge(examples, on='title_diff')
    .sort_values('distinct_ebook_titles', ascending=False)
    .reset_index(drop=True)
)



## pull out examples for the most commonly found title diffs
for _, row in diff_summary.head(20).iterrows():
    print(f"\ntitle_diff : {row['title_diff']}")
    print(f"distinct_ebook_titles : {row['distinct_ebook_titles']}  |  occurrences : {row['occurrences']}")
    print("examples: ebook → physical book")
    for ebook, book in row['examples']:
        print(f"  {ebook}  →  {book}")
    print("-" * 80)


title_diff : A
distinct_ebook_titles : 77  |  occurrences : 407
examples: ebook → physical book
  THE SEVEN HUSBANDS OF EVELYN HUGO: A NOVEL  →  CARRIE SOTO IS BACK : A NOVEL
  DEMON COPPERHEAD: A PULITZER PRIZE WINNER  →  DEMON COPPERHEAD : A NOVEL
  THE UNDERGROUND RAILROAD (OPRAH'S BOOK CLUB): A NOVEL  →  THE UNDERGROUND RAILROAD : A NOVEL
  REMARKABLY BRIGHT CREATURES: A READ WITH JENNA PICK  →  REMARKABLY BRIGHT CREATURES : A NOVEL
  THE GIRL ON THE TRAIN: A NOVEL  →  INTO THE WATER
--------------------------------------------------------------------------------

title_diff : THE
distinct_ebook_titles : 68  |  occurrences : 429
examples: ebook → physical book
  THE SEVEN HUSBANDS OF EVELYN HUGO: A NOVEL  →  CARRIE SOTO IS BACK : A NOVEL
  THE HOUSE OF BROKEN ANGELS  →  THE HOUSE OF BROKEN ANGELS : A NOVEL
  THE COVENANT OF WATER  →  THE COVENANT OF WATER : A NOVEL
  THE UNDERGROUND RAILROAD (OPRAH'S BOOK CLUB): A NOVEL  →  THE UNDERGROUND RAILROAD : A NOVEL
  THE WRONG SIDE OF GO

/var/folders/0w/2svnnw0n25lcyg2s3_xcn6rc0000gn/T/ipykernel_37437/3067042099.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  diff_with_examples


In [139]:
# test the original physical book -> ebook matching logic to see how well it works the other way

# swap inputs
fuzzy_crossed_v6_reversed = fuzzy_cross_rank_v2(
    ebook_ranked,   # now the "book_df" being iterated
    book_ranked,    # now the "ebook_df" being searched
    book_rank_cutoff=5
)

# rename columns to reflect actual content
fuzzy_crossed_v6_reversed = fuzzy_crossed_v6_reversed.rename(columns={
    'book_title':            'ebook_title',
    'ebook_title':           'book_title',
    'book_author':           'ebook_author',
    'ebook_author':          'book_author',
    'book_rank':             'ebook_rank',
    'ebook_rank':            'book_rank',
    'book_total_checkouts':  'ebook_total_checkouts',
    'ebook_total_checkouts': 'book_total_checkouts',
})

total   = len(fuzzy_crossed_v6_reversed)
matched = fuzzy_crossed_v6_reversed['book_rank'].notna().sum()

print("=== Method 6 Reversed: Fuzzy match ebook → physical book ===")
print(f"Total ebook entries:           {total:,}")
print(f"Matched in physical books:     {matched:,} ({matched/total*100:.1f}%)")
print(f"Not matched in physical books: {total-matched:,} ({(total-matched)/total*100:.1f}%)")
print()
for tier in [5, 10, 20, 50]:
    count = (fuzzy_crossed_v6_reversed['book_rank'] <= tier).sum()
    print(f"In physical book top {tier:>2}: {count:>5,} ({count/total*100:.1f}%)")

=== Method 6 Reversed: Fuzzy match ebook → physical book ===
Total ebook entries:           540
Matched in physical books:     460 (85.2%)
Not matched in physical books: 80 (14.8%)

In physical book top  5:    82 (15.2%)
In physical book top 10:   100 (18.5%)
In physical book top 20:   128 (23.7%)
In physical book top 50:   189 (35.0%)


In [141]:
# Get top 20 unmatched books by best combined score
print("=== Top 20 Unmatched EBooks (Method 6) — Top 5 Author-Matched  Physical book Candidates ===\n")

def check_unmatched_fuzzy_reverse(ebook_df, book_df, ebook_rank_cutoff=5,
                                   book_rank_cutoff=50, author_threshold=70):
    """
    For each unmatched ebook in fuzzy_crossed_v6_reversed,
    return the top 5 author-matched physical book candidates that still failed.
    """
    def clean_author(author):
        author = re.sub(r'\(.*?\)', '', str(author))
        author = re.sub(r',?\s*(JR|SR|II|III|IV)\.?', '', author, flags=re.IGNORECASE)
        author = re.sub(r',?\s*\d{4}-?(\d{4})?', '', author)
        return author.strip()

    noise = {'a', 'an', 'the', 'of', 'in', 'and', 'or', 'to', 'by', 'for',
             'novel', 'memoir', 'story', 'book', 'vol', 'volume', 'series',
             'pick', 'winner', 'prize', 'club', 'read', 'with', 'jenna',
             'oprah', 'reese', 'pulitzer', 'collection', 'continuing'}

    def meaningful_words(text):
        words = re.sub(r'[^a-z0-9\s]', ' ', str(text).lower()).split()
        return [w for w in words if w not in noise and len(w) >= 3]

    # unmatched ebook+month combinations from reversed match
    unmatched = (
        fuzzy_crossed_v6_reversed[fuzzy_crossed_v6_reversed['book_rank'].isna()]
        [['ebook_title', 'ebook_author', 'checkoutyear', 'checkoutmonth', 'ebook_rank']]
        .drop_duplicates('ebook_title')
        .nlargest(20, 'ebook_rank')   # top 20 by ebook rank
    )

    for _, ebook_row in unmatched.iterrows():
        year  = ebook_row['checkoutyear']
        month = ebook_row['checkoutmonth']

        # get physical book pool for that month
        book_month = (
            book_df[
                (book_df['checkoutyear'] == year) &
                (book_df['checkoutmonth'] == month) &
                (book_df['month_checkout_rank'] <= book_rank_cutoff)
            ]
            .drop_duplicates('title')
            .copy()
        )
        book_authors  = book_month['creator'].tolist()
        ebook_author_clean = clean_author(ebook_row['ebook_author'])
        ebook_words   = meaningful_words(ebook_row['ebook_title'])

        # fuzzy match author
        author_candidates = process.extract(ebook_author_clean, book_authors,
                                            scorer=fuzz.token_sort_ratio, limit=5)

        print(f"{'═' * 80}")
        print(f"Ebook  : {ebook_row['ebook_title']}")
        print(f"Author : {ebook_row['ebook_author']}  |  Ebook Rank: {ebook_row['ebook_rank']}")
        print(f"Month  : {year}-{month:02d}")
        print(f"Meaningful words: {ebook_words}")
        print(f"\nTop 5 author-matched physical book candidates:")

        for book_author_name, author_score, idx in author_candidates:
            candidate     = book_month.iloc[idx]
            title_score   = fuzz.WRatio(ebook_row['ebook_title'], candidate['title'])
            book_words    = meaningful_words(candidate['title'])
            # check shorter → longer word overlap
            shorter = ebook_words if len(ebook_words) <= len(book_words) else book_words
            longer  = ' '.join(book_words if len(ebook_words) <= len(book_words) else ebook_words)
            matched_words = [w for w in shorter if w in longer]
            word_overlap  = len(matched_words) / len(shorter) if shorter else 0

            print(f"  {'─' * 70}")
            print(f"  Book Title    : {candidate['title']}")
            print(f"  Book Author   : {candidate['creator']}")
            print(f"  Author Score  : {author_score:.1f}  |  Title Score: {title_score:.1f}  |  Word Overlap: {word_overlap:.0%}")
            print(f"  Matched Words : {matched_words}")
        print()


check_unmatched_fuzzy_reverse(ebook_ranked, book_ranked,
                               ebook_rank_cutoff=5,
                               book_rank_cutoff=50)

=== Top 20 Unmatched EBooks (Method 6) — Top 5 Author-Matched  Physical book Candidates ===

════════════════════════════════════════════════════════════════════════════════
Ebook  : THE TESTAMENTS: THE SEQUEL TO THE HANDMAID'S TALE
Author : MARGARET ATWOOD  |  Ebook Rank: 5
Month  : 2020-01
Meaningful words: ['testaments', 'sequel', 'handmaid', 'tale']

Top 5 author-matched physical book candidates:
  ──────────────────────────────────────────────────────────────────────
  Book Title    : THE TESTAMENTS
  Book Author   : ATWOOD, MARGARET
  Author Score  : 96.8  |  Title Score: 90.0  |  Word Overlap: 100%
  Matched Words : ['testaments']
  ──────────────────────────────────────────────────────────────────────
  Book Title    : THE YELLOW HOUSE
  Book Author   : BROOM, SARAH M.
  Author Score  : 46.7  |  Title Score: 85.5  |  Word Overlap: 0%
  Matched Words : []
  ──────────────────────────────────────────────────────────────────────
  Book Title    : BLOWOUT : CORRUPTED DEMOCRACY, ROG

In [144]:
## evaluate which option yields better results:
## among fuzzy matched closest author/ title, return title with either
## a) most meaningful words directly matched (at least 1)
## b) > 80% word overlap

def fuzzy_match_evaluate(primary_df, candidate_df, primary_label='ebook',
                          candidate_label='book', rank_cutoff=5,
                          author_threshold=70, min_substring_len=3):
    """
    For each title in primary_df, fuzzy match author then title against candidate_df.
    Returns results under two criteria:
      a) at_least_1_word : best title match with >= 1 meaningful word overlap
      b) word_overlap_80 : best title match with >= 80% meaningful word overlap (shorter → longer)
    """
    noise = {'a', 'an', 'the', 'of', 'in', 'and', 'or', 'to', 'by', 'for',
             'novel', 'memoir', 'story', 'book', 'vol', 'volume', 'series',
             'pick', 'winner', 'prize', 'club', 'read', 'with', 'jenna',
             'oprah', 'reese', 'pulitzer', 'collection', 'continuing'}

    def clean_author(author):
        author = re.sub(r'\(.*?\)', '', str(author))
        author = re.sub(r',?\s*(JR|SR|II|III|IV)\.?', '', author, flags=re.IGNORECASE)
        author = re.sub(r',?\s*\d{4}-?(\d{4})?', '', author)
        return author.strip()

    def meaningful_words(text):
        words = re.sub(r'[^a-z0-9\s]', ' ', str(text).lower()).split()
        return [w for w in words if w not in noise and len(w) >= min_substring_len]

    def word_overlap(words_a, words_b):
        """Directional: fraction of shorter title's words in longer title."""
        if not words_a or not words_b:
            return 0.0
        shorter = words_a if len(words_a) <= len(words_b) else words_b
        longer  = ' '.join(words_b if len(words_a) <= len(words_b) else words_a)
        matched = [w for w in shorter if w in longer]
        return len(matched) / len(shorter)

    results = []

    for (year, month), primary_group in (
        primary_df[primary_df['month_checkout_rank'] <= rank_cutoff]
        .groupby(['checkoutyear', 'checkoutmonth'])
    ):
        candidate_month = (
            candidate_df[
                (candidate_df['checkoutyear'] == year) &
                (candidate_df['checkoutmonth'] == month)
            ]
            .drop_duplicates('title')
            .copy()
        )
        candidate_authors = candidate_month['creator'].tolist()

        for _, primary_row in primary_group.iterrows():
            primary_author_clean = clean_author(primary_row['creator'])
            primary_words        = meaningful_words(primary_row['title'])

            # Step 1 — fuzzy author match
            author_candidates = process.extract(primary_author_clean, candidate_authors,
                                                scorer=fuzz.token_sort_ratio, limit=10)
            passing_authors = [(score, idx) for _, score, idx in author_candidates
                               if score >= author_threshold]

            if not passing_authors:
                results.append(_empty_row(primary_row, year, month,
                                          primary_label, candidate_label))
                continue

            # Step 2 — fuzzy title score for all passing authors
            title_candidates = []
            for author_score, idx in passing_authors:
                candidate   = candidate_month.iloc[idx]
                title_score = fuzz.WRatio(primary_row['title'], candidate['title'])
                cand_words  = meaningful_words(candidate['title'])
                matched     = [w for w in primary_words if w in ' '.join(cand_words)]
                overlap     = word_overlap(primary_words, cand_words)
                title_candidates.append({
                    'title_score':   title_score,
                    'author_score':  author_score,
                    'idx':           idx,
                    'matched_words': matched,
                    'overlap':       overlap,
                })

            # best overall = highest title score regardless of word criteria
            title_candidates.sort(key=lambda x: -x['title_score'])
            best_overall = title_candidates[0]
            best_candidate = candidate_month.iloc[best_overall['idx']]

            # criterion a — at least 1 meaningful word matched
            crit_a = next((c for c in title_candidates if len(c['matched_words']) >= 1), None)

            # criterion b — >= 80% word overlap
            crit_b = next((c for c in title_candidates if c['overlap'] >= 0.8), None)

            results.append({
                f'{primary_label}_title':          primary_row['title'],
                f'{primary_label}_author':         primary_row['creator'],
                f'{primary_label}_rank':           primary_row['month_checkout_rank'],
                'checkoutyear':                    year,
                'checkoutmonth':                   month,
                # best fuzzy match regardless of word criteria
                'best_title':                      best_candidate['title'],
                'best_author':                     best_candidate['creator'],
                'best_title_score':                best_overall['title_score'],
                'best_author_score':               best_overall['author_score'],
                'best_overlap':                    best_overall['overlap'],
                'best_matched_words':              str(best_overall['matched_words']),
                # criterion a
                'crit_a_title':                    candidate_month.iloc[crit_a['idx']]['title'] if crit_a else None,
                'crit_a_title_score':              crit_a['title_score']   if crit_a else None,
                'crit_a_author_score':             crit_a['author_score']  if crit_a else None,
                'crit_a_overlap':                  crit_a['overlap']       if crit_a else None,
                'crit_a_matched_words':            str(crit_a['matched_words']) if crit_a else None,
                # criterion b
                'crit_b_title':                    candidate_month.iloc[crit_b['idx']]['title'] if crit_b else None,
                'crit_b_title_score':              crit_b['title_score']   if crit_b else None,
                'crit_b_author_score':             crit_b['author_score']  if crit_b else None,
                'crit_b_overlap':                  crit_b['overlap']       if crit_b else None,
                'crit_b_matched_words':            str(crit_b['matched_words']) if crit_b else None,
            })

    return pd.DataFrame(results)


def _empty_row(primary_row, year, month, primary_label, candidate_label):
    return {
        f'{primary_label}_title':  primary_row['title'],
        f'{primary_label}_author': primary_row['creator'],
        f'{primary_label}_rank':   primary_row['month_checkout_rank'],
        'checkoutyear':            year,
        'checkoutmonth':           month,
        'best_title':              None, 'best_author':        None,
        'best_title_score':        None, 'best_author_score':  None,
        'best_overlap':            None, 'best_matched_words': None,
        'crit_a_title':            None, 'crit_a_title_score': None,
        'crit_a_author_score':     None, 'crit_a_overlap':     None,
        'crit_a_matched_words':    None,
        'crit_b_title':            None, 'crit_b_title_score': None,
        'crit_b_author_score':     None, 'crit_b_overlap':     None,
        'crit_b_matched_words':    None,
    }

# ── Run both directions ───────────────────────────────────────────────────────
results_ebook_to_book = fuzzy_match_evaluate(
    ebook_ranked, book_ranked,
    primary_label='ebook', candidate_label='book', rank_cutoff=5
)

results_book_to_ebook = fuzzy_match_evaluate(
    book_ranked, ebook_ranked,
    primary_label='book', candidate_label='ebook', rank_cutoff=5
)

# ── Match rate comparison ─────────────────────────────────────────────────────
for label, df in [('Ebook → Book', results_ebook_to_book),
                  ('Book → Ebook', results_book_to_ebook)]:
    total   = len(df)
    a_match = df['crit_a_title'].notna().sum()
    b_match = df['crit_b_title'].notna().sum()
    print(f"=== {label} ===")
    print(f"  Total entries              : {total:,}")
    print(f"  Crit A (>= 1 word matched) : {a_match:,} ({a_match/total*100:.1f}%)")
    print(f"  Crit B (>= 80% overlap)    : {b_match:,} ({b_match/total*100:.1f}%)")
    print()

=== Ebook → Book ===
  Total entries              : 540
  Crit A (>= 1 word matched) : 488 (90.4%)
  Crit B (>= 80% overlap)    : 485 (89.8%)

=== Book → Ebook ===
  Total entries              : 540
  Crit A (>= 1 word matched) : 503 (93.1%)
  Crit B (>= 80% overlap)    : 496 (91.9%)



In [145]:
# ── Highest scoring unmatched — crit A vs crit B ─────────────────────────────
for label, df, primary_label in [
    ('Ebook → Book', results_ebook_to_book, 'ebook'),
    ('Book → Ebook', results_book_to_ebook, 'book')
]:
    print(f"\n{'═'*80}")
    print(f"=== {label} — Top 10 unmatched by crit A (highest title score that still failed) ===")
    unmatched_a = (
        df[df['crit_a_title'].isna() & df['best_title'].notna()]
        .drop_duplicates(f'{primary_label}_title')
        .nlargest(10, 'best_title_score')
        [[f'{primary_label}_title', 'best_title', 'best_author_score',
          'best_title_score', 'best_overlap', 'best_matched_words']]
    )
    print(unmatched_a.to_string())

    print(f"\n=== {label} — Top 10 unmatched by crit B (highest title score that still failed) ===")
    unmatched_b = (
        df[df['crit_b_title'].isna() & df['best_title'].notna()]
        .drop_duplicates(f'{primary_label}_title')
        .nlargest(10, 'best_title_score')
        [[f'{primary_label}_title', 'best_title', 'best_author_score',
          'best_title_score', 'best_overlap', 'best_matched_words']]
    )
    print(unmatched_b.to_string())


════════════════════════════════════════════════════════════════════════════════
=== Ebook → Book — Top 10 unmatched by crit A (highest title score that still failed) ===
                                                                                      ebook_title                                                                                     best_title  best_author_score  best_title_score  best_overlap best_matched_words
249                                                                       THE GUEST LIST: A NOVEL  RESILIENT GRIEVING : FINDING STRENGTH AND EMBRACING LIFE AFTER A LOSS THAT CHANGES EVERYTHING          70.000000             85.50           0.0                 []
256  THE ART OF TAKING IT EASY: HOW TO COPE WITH BEARS, TRAFFIC, AND THE REST OF LIFE'S STRESSORS                                                     YELLOWSTONE AND GRAND TETON NATIONAL PARKS          81.818182             85.50           0.0                 []
431                                    

In [146]:
# ── Lowest scoring matched — crit A vs crit B ────────────────────────────────
for label, df, primary_label in [
    ('Ebook → Book', results_ebook_to_book, 'ebook'),
    ('Book → Ebook', results_book_to_ebook, 'book')
]:
    print(f"\n{'═'*80}")
    print(f"=== {label} — Bottom 10 matched by crit A (lowest title score) ===")
    matched_a = (
        df[df['crit_a_title'].notna()]
        .drop_duplicates(f'{primary_label}_title')
        .nsmallest(10, 'crit_a_title_score')
        [[f'{primary_label}_title', 'crit_a_title', 'crit_a_author_score',
          'crit_a_title_score', 'crit_a_overlap', 'crit_a_matched_words']]
    )
    print(matched_a.to_string())

    print(f"\n=== {label} — Bottom 10 matched by crit B (lowest title score) ===")
    matched_b = (
        df[df['crit_b_title'].notna()]
        .drop_duplicates(f'{primary_label}_title')
        .nsmallest(10, 'crit_b_title_score')
        [[f'{primary_label}_title', 'crit_b_title', 'crit_b_author_score',
          'crit_b_title_score', 'crit_b_overlap', 'crit_b_matched_words']]
    )
    print(matched_b.to_string())


════════════════════════════════════════════════════════════════════════════════
=== Ebook → Book — Bottom 10 matched by crit A (lowest title score) ===
                                                                                                            ebook_title                             crit_a_title  crit_a_author_score  crit_a_title_score  crit_a_overlap               crit_a_matched_words
472                                                                               MEXIKID: (NEWBERY HONOR AWARD WINNER)                       MEXIKID EN ESPAÑOL            88.000000           60.000000             0.5                        ['mexikid']
297                                                                                        THE LINCOLN HIGHWAY: A NOVEL                      THE LINCOLN HIGHWAY            95.652174           80.851064             1.0             ['lincoln', 'highway']
241                                                                                    

In [147]:
# Ebook → Book: in Crit A but not Crit B
ebook_a_not_b = results_ebook_to_book[
    results_ebook_to_book['crit_a_title'].notna() &
    results_ebook_to_book['crit_b_title'].isna()
].drop_duplicates('ebook_title')

print("=== Ebook → Book: matched by Crit A but not Crit B ===")
print(ebook_a_not_b[['ebook_title', 'crit_a_title', 'crit_a_author_score',
                      'crit_a_title_score', 'crit_a_overlap',
                      'crit_a_matched_words']].to_string())

print()

# Book → Ebook: in Crit A but not Crit B
book_a_not_b = results_book_to_ebook[
    results_book_to_ebook['crit_a_title'].notna() &
    results_book_to_ebook['crit_b_title'].isna()
].drop_duplicates('book_title')

print("=== Book → Ebook: matched by Crit A but not Crit B ===")
print(book_a_not_b[['book_title', 'crit_a_title', 'crit_a_author_score',
                     'crit_a_title_score', 'crit_a_overlap',
                     'crit_a_matched_words']].to_string())

=== Ebook → Book: matched by Crit A but not Crit B ===
                                                                                                            ebook_title                             crit_a_title  crit_a_author_score  crit_a_title_score  crit_a_overlap crit_a_matched_words
12   HARRY POTTER AND THE CURSED CHILD: PARTS ONE AND TWO: THE OFFICIAL SCRIPT BOOK OF THE ORIGINAL WEST END PRODUCTION  HARRY POTTER AND THE CHAMBER OF SECRETS            96.296296                85.5             0.5  ['harry', 'potter']
472                                                                               MEXIKID: (NEWBERY HONOR AWARD WINNER)                       MEXIKID EN ESPAÑOL            88.000000                60.0             0.5          ['mexikid']

=== Book → Ebook: matched by Crit A but not Crit B ===
                                                                                                                                                       book_title           

#### using word overlap > .8 threshold over >1 meaningful word matched returns more accurate results
#### replace that criteria in fuzzy match function

### Generalize fuzzy matching to be bi-directional (ebook -> book, book -> ebook)

In [155]:
## generalize matching logic to be direction agnostic: handle ebook -> book, book -> ebook

def fuzzy_cross_rank(origin_df, destination_df,
                     origin_label='origin', destination_label='destination',
                     origin_rank_cutoff=10, destination_rank_cutoff=None,
                     author_threshold=70, min_substring_len=3,
                     overlap_threshold=0.8):
    """
    Direction-agnostic fuzzy matching function.
    Matches titles in origin_df to closest title in destination_df.
    Title matching criterion: >= overlap_threshold word overlap (shorter → longer).

    Parameters:
        origin_df               : DataFrame being iterated
        destination_df          : DataFrame being searched
        origin_label            : label for origin columns (e.g. 'book' or 'ebook')
        destination_label       : label for destination columns (e.g. 'ebook' or 'book')
        origin_rank_cutoff      : only consider origin titles ranked <= this value
        destination_rank_cutoff : only search destination titles ranked <= this value (None = no filter)
        author_threshold        : minimum fuzzy author score to consider
        min_substring_len       : minimum word length to be considered meaningful
        overlap_threshold       : minimum word overlap fraction to accept a title match (default 0.8)
    """

    noise = {'a', 'an', 'the', 'of', 'in', 'and', 'or', 'to', 'by', 'for',
             'novel', 'memoir', 'story', 'book', 'vol', 'volume', 'series',
             'pick', 'winner', 'prize', 'club', 'read', 'with', 'jenna',
             'oprah', 'reese', 'pulitzer', 'collection', 'continuing'}

    def clean_author(author):
        author = re.sub(r'\(.*?\)', '', str(author))
        author = re.sub(r',?\s*(JR|SR|II|III|IV)\.?', '', author, flags=re.IGNORECASE)
        author = re.sub(r',?\s*\d{4}-?(\d{4})?', '', author)
        return author.strip()

    def meaningful_words(title, author=None):
        words = set(re.sub(r'[^a-z0-9\s]', ' ', str(title).lower()).split())
        author_words = set(re.sub(r'[^a-z0-9\s]', ' ', str(author).lower()).split()) if author else set()
        return [w for w in words
                if w not in noise
                and len(w) >= min_substring_len
                and w not in author_words]

    def word_overlap(words_a, words_b):
        """Directional: fraction of shorter title's words found in longer title."""
        if not words_a or not words_b:
            return 0.0
        shorter = words_a if len(words_a) <= len(words_b) else words_b
        longer  = ' '.join(words_b if len(words_a) <= len(words_b) else words_a)
        matched = [w for w in shorter if w in longer]
        return len(matched) / len(shorter)

    results = []

    for (year, month), origin_group in (
        origin_df[origin_df['month_checkout_rank'] <= origin_rank_cutoff]
        .groupby(['checkoutyear', 'checkoutmonth'])
    ):
        dest_month = destination_df[
            (destination_df['checkoutyear'] == year) &
            (destination_df['checkoutmonth'] == month)
        ]
        if destination_rank_cutoff is not None:
            dest_month = dest_month[dest_month['month_checkout_rank'] <= destination_rank_cutoff]

        dest_month   = dest_month.drop_duplicates('title').copy()
        dest_authors = dest_month['creator'].tolist()

        for _, origin_row in origin_group.iterrows():
            origin_author_clean = clean_author(origin_row['creator'])
            origin_words        = meaningful_words(origin_row['title'], origin_row['creator'])

            # Step 1 — fuzzy author match
            author_candidates = process.extract(origin_author_clean, dest_authors,
                                                scorer=fuzz.token_sort_ratio, limit=10)
            passing_authors = [(score, idx) for _, score, idx in author_candidates
                               if score >= author_threshold]

            best_match = best_title_score = best_author_score = best_matched_words = best_overlap = None

            if passing_authors:
                # Step 2 — among passing authors, find candidates meeting overlap threshold
                title_candidates = []

                for author_score, idx in passing_authors:
                    candidate     = dest_month.iloc[idx]
                    dest_words    = meaningful_words(candidate['title'])
                    overlap       = word_overlap(origin_words, dest_words)
                    matched_words = [w for w in (origin_words if len(origin_words) <= len(dest_words)
                                                 else dest_words)
                                     if w in ' '.join(dest_words if len(origin_words) <= len(dest_words)
                                                      else origin_words)]

                    # return titles that meet overlap_threshold
                    if overlap >= overlap_threshold:
                        title_score = fuzz.WRatio(origin_row['title'], candidate['title'])
                        title_candidates.append((title_score, author_score,
                                                 overlap, idx, matched_words))

                if title_candidates:
                    title_candidates.sort(key=lambda x: x[0], reverse=True)
                    best_title_score, best_author_score, best_overlap, idx, best_matched_words = title_candidates[0]
                    best_match = dest_month.iloc[idx]

            if best_match is not None:
                results.append({
                    'checkoutyear':                          year,
                    'checkoutmonth':                         month,
                    'month_date':                            origin_row.get('month_date'),
                    f'{origin_label}_title':                 origin_row['title'],
                    f'{origin_label}_author':                origin_row['creator'],
                    f'{origin_label}_rank':                  origin_row['month_checkout_rank'],
                    f'{origin_label}_total_checkouts':       origin_row.get('total_checkouts'),
                    f'{destination_label}_title':            best_match['title'],
                    f'{destination_label}_author':           best_match['creator'],
                    f'{destination_label}_rank':             best_match['month_checkout_rank'],
                    f'{destination_label}_total_checkouts':  best_match.get('total_checkouts'),
                    'title_score':                           best_title_score,
                    'author_score':                          best_author_score,
                    'word_overlap':                          best_overlap,
                    'matched_words':                         str(best_matched_words),
                    'genre':                                 origin_row.get('genre'),
                    'match_found':                           True,
                })
            else:
                results.append({
                    'checkoutyear':                          year,
                    'checkoutmonth':                         month,
                    'month_date':                            origin_row.get('month_date'),
                    f'{origin_label}_title':                 origin_row['title'],
                    f'{origin_label}_author':                origin_row['creator'],
                    f'{origin_label}_rank':                  origin_row['month_checkout_rank'],
                    f'{origin_label}_total_checkouts':       origin_row.get('total_checkouts'),
                    f'{destination_label}_title':            None,
                    f'{destination_label}_author':           None,
                    f'{destination_label}_rank':             None,
                    f'{destination_label}_total_checkouts':  None,
                    'title_score':                           None,
                    'author_score':                          None,
                    'word_overlap':                          None,
                    'matched_words':                         None,
                    'genre':                                 origin_row.get('genre'),
                    'match_found':                           False,
                })

    return pd.DataFrame(results)



def print_match_report(df, origin_label, destination_label, rank_cutoff):
    total   = len(df)
    matched = df['match_found'].sum()
    print(f"=== Fuzzy Match: {origin_label} → {destination_label} (top {rank_cutoff}) ===")
    print(f"  Total entries    : {total:,}")
    print(f"  Matched          : {matched:,} ({matched/total*100:.1f}%)")
    print(f"  Not matched      : {total-matched:,} ({(total-matched)/total*100:.1f}%)")
    print()
    for tier in [5, 10, 20, 50]:
        count = (df[f'{destination_label}_rank'] <= tier).sum()
        print(f"  {destination_label} top {tier:>2}: {count:>5,} ({count/total*100:.1f}%)")

In [159]:
# ── Book → Ebook 
book_to_ebook = fuzzy_cross_rank(
    book_ranked, ebook_ranked,
    origin_label='book', destination_label='ebook',
    origin_rank_cutoff=10
)
print_match_report(book_to_ebook, 'book', 'ebook', rank_cutoff=10)

# ── Ebook → Book 
ebook_to_book = fuzzy_cross_rank(
    ebook_ranked, book_ranked,
    origin_label='ebook', destination_label='book',
    origin_rank_cutoff=10
)
print_match_report(ebook_to_book, 'ebook', 'book', rank_cutoff=10)

=== Fuzzy Match: book → ebook (top 10) ===
  Total entries    : 1,080
  Matched          : 983 (91.0%)
  Not matched      : 97 (9.0%)

  ebook top  5:   105 (9.7%)
  ebook top 10:   148 (13.7%)
  ebook top 20:   273 (25.3%)
  ebook top 50:   441 (40.8%)
=== Fuzzy Match: ebook → book (top 10) ===
  Total entries    : 1,080
  Matched          : 941 (87.1%)
  Not matched      : 139 (12.9%)

  book top  5:   114 (10.6%)
  book top 10:   143 (13.2%)
  book top 20:   192 (17.8%)
  book top 50:   321 (29.7%)
